# D41 Data Engineering Code

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 사내 문서 수집과 합성 데이터 실습

## 실행 환경

**실습 목표**
- 6종 문서 형식과 인코딩 오류·중복·개인정보가 혼재된 사내 문서 저장소에서 카탈로그를 생성하고 인코딩 폴백 및 추출 품질 지표를 산출할 수 있다.
- 문서·줄 단위 정제 규칙과 MinHash LSH·TF-IDF·SimHash 근사 중복 검출을 수행하여 제외 사유가 기록된 정제 스냅샷을 생성할 수 있다.
- 정제된 문서를 기반으로 평가용 질의응답(QA) 합성 데이터를 생성하고 형식·근거·수치 일관성 검증 함수를 구현할 수 있다.

<div style="font-size: 18px; line-height: 1.6;">

| 항목 | 세부 내용 |
|---|---|
| **기준 실행 환경** | Google Colab Pro+ (`T4`, `L4`, `A100` GPU 모두 호환) |
| **외부 API 키** | 불필요 (모든 파서와 모델을 로컬 환경에서 직접 실행함) |
| **로드하는 모델** | `Qwen/Qwen2.5-3B-Instruct` 1종 (4비트 양자화 로드, VRAM 약 2.5GB 사용) |
| **CPU 단독 실행 범위** | Step 0 ~ Step 8 전 과정 (GPU는 Step 9 합성 데이터 생성부터 필요함) |
| **주요 단계별 소요 시간** | Step 0 패키지 설치 3~4분, Step 9 모델 로드 3~5분, Step 9 문서당 생성 20~40초 |
| **전체 예상 소요 시간** | 약 4~5시간 (선택 심화 트랙 2개 수행 포함) |

</div>

오늘 실습의 대부분은 파일 입출력, 정규식 처리, 통계 집계이므로 GPU가 없는 런타임에서도 Step 8까지 그대로 실행됩니다. **데이터 엔지니어링의 핵심인 추출·정제·중복 제거·품질 계측은 대부분 CPU 연산으로 수행됩니다.**

## Step 0. 환경 준비

이론 3.2절과 3.8절에서 살펴본 것처럼 문서 파서와 전처리 라이브러리는 마이너 버전 업데이트만으로도 공백 처리나 추출 결과가 달라질 수 있으므로 메이저 버전 상한을 지정하는 방식(`"패키지>=6,<7"`)으로 호환 패치는 허용하되 파괴적 변경은 차단합니다.

In [5]:
import torch
from google.colab import drive
import os

drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/aiffel
os.chdir('/content/drive/MyDrive/aiffel')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# 문서 처리 쪽
!pip install -q "pypdf>=6,<7" "python-hwpx>=6,<7" "python-docx>=1.2,<2" \
                "openpyxl>=3.1,<4" "fpdf2>=2.8,<3"
# 정제와 중복 판정 쪽
!pip install -q "chardet>=7,<8" "datasketch>=2,<3"
# 생성 모델 쪽 (Step 9 부터 씁니다)
!pip install -q "transformers>=5.17,<6" "accelerate>=1.15,<2" "bitsandbytes>=0.50,<0.51"
# 선택 심화 트랙 1(임베딩 기반 근사 중복)에서 사용합니다
!pip install -q "langchain-huggingface>=1.2,<2" "sentence-transformers>=6,<7"

`transformers`를 고정하면 함께 설치되는 의존성 패키지의 버전이 바뀌는 일이 있어 설치가 끝나면 코랩이 **런타임을 다시 시작하라**고 알릴 수 있습니다. 알림이 뜨면 다시 시작한 뒤 설치 셀은 건너뛰고 이어서 실행합니다.

In [8]:
# 실제로 설치된 버전을 출력해 기록해 둡니다. 나중에 결과가 달라졌을 때 가장 먼저 볼 기록입니다
from importlib.metadata import version, PackageNotFoundError

PINNED = ["pypdf", "python-hwpx", "python-docx", "openpyxl", "fpdf2",
          "chardet", "datasketch", "transformers", "accelerate", "bitsandbytes"]
for name in PINNED:
    try:
        print(f"{name:16} {version(name)}")
    except PackageNotFoundError:
        print(f"{name:16} (설치되지 않음)")

pypdf            6.21.1
python-hwpx      6.8.0
python-docx      1.2.0
openpyxl         3.1.5
fpdf2            2.8.9
chardet          7.7.0
datasketch       2.0.0
transformers     5.20.1
accelerate       1.15.0
bitsandbytes     0.50.2


`__version__` 속성 대신 `importlib.metadata.version()`을 썼습니다. **패키지 중에는 `__version__`을 아예 두지 않는 것이 있지만** 설치 정보에서 읽으면 어떤 패키지든 같은 방법으로 확인됩니다.

In [9]:
import os, re, json, hashlib, shutil, random, logging
from collections import Counter
from pathlib import Path
import numpy as np
import torch

random.seed(42)
np.random.seed(42)

# fpdf2 가 글꼴을 다듬으면서 내는 안내 줄을 줄입니다. 오류가 아닙니다
logging.getLogger("fontTools").setLevel(logging.ERROR)

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU : {torch.cuda.get_device_name(0)}  VRAM {props.total_memory/1e9:.1f} GB")
else:
    print("GPU 없음. Step 8 까지는 그대로 실행됩니다.")
    print("Step 9 부터는 런타임 유형을 GPU로 바꾼 뒤 처음부터 다시 실행하세요.")

GPU : Tesla T4  VRAM 15.6 GB


`torch.cuda.get_device_name(0)`을 조건 없이 호출하면 **GPU가 없는 런타임에서 첫 셀부터 예외가 납니다.** 오늘 분량의 대부분은 GPU 없이 실행되니 GPU가 있는지 먼저 묻고 없으면 안내만 합니다. 이론 전체에서 반복되는 **실패를 예외가 아니라 정상 경로로 다루기** 원칙의 코드 버전입니다.

## Step 1. 사내 문서 저장소 표본 구성

이론 1.2절의 다섯 가지 문제를 모두 심어 둔 저장소를 만듭니다. **문제를 알고 만들어야 해결책이 동작하는지 검증할 수 있습니다.**

- **형식 난립** : md, txt, docx, xlsx, pdf, hwpx 여섯 가지  
- **버전 혼재** : 같은 규정의 2024년판과 2026년판  
- **완전 중복** : 다른 폴더에 글자까지 같은 파일  
- **근사 중복** : 한 조항만 늘어난 문서  
- **민감 정보** : 주민등록번호, 연락처, 계좌번호  
- **권한 구분** : 부서별 폴더  
- **빈 문서와 템플릿과 스캔본과 인코딩이 깨진 파일**

In [10]:
ROOT = Path("./corp_docs")
if ROOT.exists():
    shutil.rmtree(ROOT)

for d in ["공용/규정", "공용/매뉴얼", "인사/평가", "인사/급여", "영업/계약", "임시"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)


def show_tree(root):
    for p in sorted(root.rglob("*")):
        rel = p.relative_to(root)
        mark = "📁" if p.is_dir() else "  "
        size = "" if p.is_dir() else f"{p.stat().st_size:>8,} B"
        print(f"{mark} {str(rel):<44} {size}")


show_tree(ROOT)

📁 공용                                       
📁 공용/규정                                 
📁 공용/매뉴얼                               
📁 영업                                       
📁 영업/계약                                 
📁 인사                                        
📁 인사/급여                                  
📁 인사/평가                                  
📁 임시                                        


폴더를 훑는 데 `!find` 같은 셸 명령 대신 파이썬을 썼습니다. 셸 명령은 코랩에서는 잘 되어도 **회사에서 쓸 환경에서는 다를 수 있으므로** 파이프라인 코드에는 되도록 셸 의존을 넣지 않습니다.

In [11]:
# --------------------------------------------------- 마크다운 문서 본문들
POLICY_V1 = """# 임직원 복무 규정 (2024년판)

## 제1조 (목적)
이 규정은 임직원의 복무에 관한 기본 사항을 정함을 목적으로 한다.

## 제3조 (근무 시간)
기본 근무 시간은 주 40시간으로 하며, 시업 시각은 오전 9시로 한다.

## 제5조 (재택근무)
임직원은 주 1회 이내에서 재택근무를 신청할 수 있다. 신청은 부서장의 승인을 받아야 한다.

## 제7조 (연차 휴가의 이월)
당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없다.

## 제9조 (시행일)
이 규정은 공포한 날부터 시행한다.
"""

POLICY_V2 = """# 임직원 복무 규정 (2026년판)

## 제1조 (목적)
이 규정은 임직원의 복무에 관한 기본 사항을 정함을 목적으로 한다.

## 제3조 (근무 시간)
기본 근무 시간은 주 40시간으로 하며, 시업 시각은 오전 10시로 한다.

## 제5조 (재택근무)
임직원은 주 2회 이내에서 재택근무를 신청할 수 있다. 신청은 근무일 전일 오후 6시까지 부서장의 승인을 받아야 한다.

## 제7조 (연차 휴가의 이월)
당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없다. 다만 업무상 사유로 사용하지 못한 경우 인사팀장의 승인을 받아 5일까지 이월할 수 있다.

## 제9조 (시행일)
이 규정은 공포한 날부터 시행한다.
"""

SECURITY = """# 정보보안 지침

## 1. 사내 데이터의 외부 반출
고객 개인정보, 미공개 재무 정보, 기술 설계 문서는 정보보호팀장의 사전 승인 없이 외부로 반출할 수 없다.
외부 AI 서비스에 사내 문서를 입력하는 행위도 반출로 간주한다.

## 2. 반출 승인 절차
반출이 필요한 경우 사내 포털의 보안 요청 메뉴에서 승인을 신청한다. 승인 없이 반출한 사실이 확인되면 즉시 보고한다.

## 3. 위반 시 조치
위반한 임직원은 상벌 규정에 따라 징계할 수 있다.
"""

MANUAL = """# 사내 시스템 사용 매뉴얼

## 1. 계정 발급
입사 후 IT팀에 계정 발급을 신청한다. 신청은 사내 포털의 IT 요청 메뉴에서 진행한다.

## 2. VPN 접속
사외에서 사내 시스템에 접속할 때는 VPN을 사용한다. VPN 클라이언트는 사내 포털에서 다운로드한다.

## 3. 문서 저장 규칙
업무 문서는 부서별 공유 드라이브에 저장한다. 개인 PC에만 저장하는 것을 금지한다.
"""

# 연락처와 이메일만 있고 계좌번호는 없는 문서. Step 5에서 쓰입니다
ONCALL = """# 야간 휴일 비상 연락망

시스템 장애가 발생하면 아래 순서로 연락합니다.

1. 1차 인프라팀 당직 - 010-2345-6789, infra-oncall@ipaltech.example.com
2. 2차 개발1팀 팀장 - 010-3456-7890, dev1-lead@ipaltech.example.com
3. 3차 정보보호팀 - 010-4567-8901, security@ipaltech.example.com

연락이 닿지 않으면 사내 포털의 장애 신고 메뉴에 기록을 남깁니다.
"""

HR_DOC = """# 2026년 상반기 인사평가 결과 (대외비)

평가 대상 및 결과

1. 김서준 (개발1팀) - 주민등록번호 880315-1234567, 연락처 010-2345-6789
   평가 등급 A, 연봉 조정률 8퍼센트, 급여 계좌 신한 110-234-567890

2. 이지우 (개발2팀) - 주민등록번호 920708-2345678, 연락처 010-3456-7890
   평가 등급 B, 연봉 조정률 4퍼센트, 급여 계좌 국민 123-45-6789012

3. 박민재 (인프라팀) - 주민등록번호 900122-1456789, 연락처 010-4567-8901
   평가 등급 A, 연봉 조정률 7퍼센트
"""

CONTRACT = """# 협력사 계약 현황

1. 주식회사 한빛물산 - 계약금액 48,000,000원, 납기 2026년 11월 30일, 담당 김서준
2. 대성테크 - 계약금액 32,000,000원, 납기 2026년 12월 15일, 담당 최유나
3. 미래솔루션 - 계약금액 55,000,000원, 납기 2027년 1월 20일, 담당 정하윤
"""

TEMPLATE = """# 제목

작성부서 :
작성자 :
작성일 :
문서번호 :
보존연한 :

1. 개요
내용을 입력하세요.

2. 세부 사항
내용을 입력하세요.

(주)아이펠테크 사내 문서 양식 v3
"""

print("본문 상수 준비 완료")

본문 상수 준비 완료


In [12]:
def w(path, text):
    p = ROOT / path
    p.write_text(text, encoding="utf-8")
    return p


# 1) 버전 혼재와 완전 중복
w("공용/규정/복무규정_2024.md", POLICY_V1)
w("공용/규정/복무규정_최종.md", POLICY_V2)
w("공용/규정/복무규정_최종_v2.md", POLICY_V2)          # 완전 중복

# 2) 완전 중복 : 다른 폴더에 같은 파일
w("공용/매뉴얼/시스템매뉴얼.md", MANUAL)
w("임시/시스템매뉴얼_복사.md", MANUAL)                  # 완전 중복

# 3) 근사 중복 : 한 조항이 늘어난 문서
w("공용/규정/보안지침_1월.md", SECURITY)
w("공용/규정/보안지침_7월.md", SECURITY.replace(
    "## 3. 위반 시 조치",
    "## 3. 반출 이력 보관\n반출 이력은 6개월간 보관한다.\n\n## 4. 위반 시 조치"))

# 4) 민감 정보가 든 문서들
w("인사/평가/2026상반기_인사평가.md", HR_DOC)
w("영업/계약/협력사계약현황.md", CONTRACT)
w("공용/매뉴얼/비상연락망.md", ONCALL)

# 5) 빈 문서와 템플릿
w("임시/신규문서.md", "")
w("임시/템플릿.md", TEMPLATE)

print("마크다운 문서 생성 완료")

마크다운 문서 생성 완료


In [13]:
# 6) 워드와 엑셀
from docx import Document as Docx

doc = Docx()
doc.add_heading("출장 여비 규정", 0)
doc.add_paragraph("제1조 (적용 범위) 이 규정은 회사 업무로 출장하는 모든 임직원에게 적용한다.")
doc.add_paragraph("제2조 (국내 출장) 국내 출장 시 일비 3만원, 숙박비는 실비로 지급한다.")
doc.add_paragraph("제3조 (해외 출장) 해외 출장 시 지역별 기준에 따라 일비를 차등 지급한다.")
doc.add_paragraph("제4조 (정산) 출장 종료 후 7일 이내에 영수증을 첨부하여 정산을 신청한다.")
doc.add_paragraph("제5조 (예외) 부서장이 사전에 승인한 경우 한도를 초과하여 지급할 수 있다.")
doc.save(ROOT / "공용/규정/출장여비규정.docx")

import openpyxl

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "부서별 예산"
ws.append(["부서", "연간예산(백만원)", "집행률"])
for row in [["개발1팀", 1200, "82%"], ["개발2팀", 980, "75%"],
            ["인프라팀", 1500, "91%"], ["영업팀", 700, "68%"]]:
    ws.append(row)
wb.save(ROOT / "공용/부서별예산.xlsx")

print("docx, xlsx 생성 완료")

docx, xlsx 생성 완료


### Step 1-1. 한글 폰트 내장 PDF 생성

이론 2절에서 다룬 PDF 텍스트 추출 과정을 실습하기 위해 실습용 PDF 문서를 직접 생성합니다. **PDF 기본 내장 글꼴에는 한글 글리프가 포함되어 있지 않아서** 한글 폰트를 명시적으로 임베딩하지 않으면 모든 글자가 가운뎃점이나 물음표로 렌더링되고 추출 결과 역시 예외 없이 깨진 문자열로 반환되므로(이론 2.7절 조용한 실패) 나눔고딕 한글 폰트 파일을 직접 등록해 생성합니다.

In [14]:
# 🖥 로컬 작업 : 사내 리눅스 서버에서는 apt 대신 글꼴 파일을 직접 배포하는 경우가 많습니다.
#               경로만 바꾸면 아래 코드는 그대로 실행됩니다.
!apt-get install -qq fonts-nanum > /dev/null 2>&1

# 글꼴 경로를 하드코딩하지 않고 있는 것을 찾아 씁니다. 환경마다 경로가 다릅니다
FONT_CANDIDATES = [
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
]
KO_FONT = next((p for p in FONT_CANDIDATES if Path(p).exists()), None)
print("한글 글꼴 :", KO_FONT)
if KO_FONT is None:
    print("!! 한글 글꼴을 찾지 못했습니다. 아래 PDF 생성 셀은 건너뛰고 넘어가세요.")

한글 글꼴 : /usr/share/fonts/truetype/nanum/NanumGothic.ttf


In [15]:
from fpdf import FPDF
from fpdf.enums import XPos, YPos

PDF_DOC = """교육비 지원 규정 (2026년판)

제1조 (목적)
이 규정은 임직원의 직무 능력 향상을 위한 교육비 지원의 기준을 정함을 목적으로 한다.

제2조 (지원 대상)
재직 기간 6개월 이상인 임직원을 대상으로 한다. 수습 기간 중인 자는 제외한다.

제3조 (지원 한도)
1인당 연간 200만원을 한도로 하며, 어학 과정은 연간 100만원을 한도로 한다.

제4조 (신청 절차)
교육 시작 14일 전까지 사내 포털의 교육 신청 메뉴에서 신청하고 부서장의 승인을 받는다.

제5조 (환수)
교육 수료 후 1년 이내에 퇴직하는 경우 지원액의 50퍼센트를 환수한다.
"""


def make_text_pdf(out_path, body, font_path):
    pdf = FPDF()
    pdf.add_page()
    pdf.add_font("ko", "", font_path)          # 한글 글꼴을 이름 "ko" 로 등록
    pdf.set_font("ko", size=12)
    for line in body.split("\n"):
        # new_x, new_y 를 주지 않으면 둘째 줄에서 예외가 납니다
        pdf.multi_cell(0, 8, line, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.output(str(out_path))
    return out_path


if KO_FONT:
    make_text_pdf(ROOT / "공용/규정/교육비지원규정.pdf", PDF_DOC, KO_FONT)
    print("교육비지원규정.pdf 생성 완료")

교육비지원규정.pdf 생성 완료


`multi_cell`을 호출할 때 `new_x=XPos.LMARGIN, new_y=YPos.NEXT`를 주었습니다. 기본값은 커서를 쓴 글의 오른쪽에 두어서 너비 `0`으로 둘째 줄을 쓰면 남은 너비가 없어 예외가 납니다.

### Step 1-2. 이미지 기반 스캔본 PDF 생성

이론 2.5절에서 다룬 **이미지 기반 스캔본 PDF**(텍스트 레이어 없이 종이를 촬영한 이미지만 포함된 PDF)를 생성합니다. 일반 PDF 파서로 스캔본을 읽으면 예외 대신 빈 문자열(`""`)이 반환되어 파이프라인을 그대로 통과하는 현상을 Step 4의 품질 지표로 직접 확인합니다.

In [16]:
from PIL import Image, ImageDraw, ImageFont

SCAN_LINES = [
    "2018년도 시설 이용 계약서",
    "",
    "제1조 (목적) 본 계약은 시설 이용에 관한 사항을 정한다.",
    "제2조 (기간) 2018년 3월 1일부터 2019년 2월 28일까지로 한다.",
    "제3조 (이용료) 월 1,200,000원으로 하고 매월 말일에 지급한다.",
    "",
    "        (주)아이펠테크          한빛물산(주)",
]


def make_scan_pdf(out_path, lines, font_path):
    """종이를 찍은 것처럼 이미지 한 장만 든 PDF를 만듭니다"""
    img = Image.new("L", (1240, 1754), 235)          # A4 비율, 약간 누런 바탕
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype(font_path, 34)
    y = 150
    for line in lines:
        draw.text((110, y), line, font=font, fill=45)
        y += 62
    draw.ellipse((980, 1330, 1140, 1490), outline=95, width=6)   # 도장 자국
    img = img.rotate(0.6, fillcolor=235)                          # 살짝 기울여 스캔
    img.save("scan_page.png")

    pdf = FPDF(format="A4")
    pdf.add_page()
    pdf.image("scan_page.png", x=0, y=0, w=210)      # 글자가 아니라 그림으로 넣습니다
    pdf.output(str(out_path))
    return out_path


if KO_FONT:
    make_scan_pdf(ROOT / "영업/계약/시설이용계약서_스캔본.pdf", SCAN_LINES, KO_FONT)
    print("시설이용계약서_스캔본.pdf 생성 완료")

시설이용계약서_스캔본.pdf 생성 완료


### Step 1-3. 한글 문서(.hwpx) 표본 생성

이론 3.7절에서 살펴본 것처럼 국내 기업·기관 저장소에서는 한글 문서를 반드시 다루게 되며, 구형 이진 포맷인 `.hwp`와 개방형 XML 표준(OWPML)인 `.hwpx`는 내부 구조가 완전히 다른 포맷입니다. 오늘 생성하고 파싱하는 `.hwpx`는 ZIP 압축 아카이브 내부에 XML이 담긴 구조여서 `python-hwpx` 같은 순수 파이썬 라이브러리만으로 리눅스 폐쇄망 서버에서도 외부 통신 없이 문단과 표를 정확히 추출할 수 있습니다.

**[현업 환경 적용]** : 구형 `.hwp` 파일은 오늘 코랩 실습에 직접 포함하지 않았습니다. 이론 3.7절에서 정리한 것처럼 순수 파이썬 기반 `.hwp` 파서(`pyhwp`, `libhwp`)는 수년째 유지보수가 중단되어 최신 한글 버전의 표·글상자 호환성을 보장하기 어렵고 활발히 관리되는 `pyhwpx`는 윈도우 한글(HWP) 프로그램을 COM 오토메이션으로 제어하는 방식이어서 리눅스 코랩 환경에서는 실행되지 않기 때문입니다.

따라서 사내 리눅스 서버에 파이프라인을 구축할 때는 구형 `.hwp`를 PDF로 변환하지 말고(표 셀 구조 소실 방지), 수집 전 단계에서 변환 워커를 통해 `.hwpx`로 일괄 표준화한 뒤 본 노트의 `python-hwpx` 코드로 처리합니다.

In [17]:
from hwpx.document import HwpxDocument

HWPX_LINES = [
    "공문 접수 및 처리 지침",
    "",
    "제1조 (접수) 외부 기관에서 온 공문은 총무팀이 일괄 접수한다.",
    "제2조 (분류) 접수한 공문은 소관 부서를 확인하여 분류한다.",
    "제3조 (처리 기한) 접수일로부터 3일 이내에 담당 부서로 전달한다.",
    "제4조 (회신) 회신이 필요한 공문은 접수일로부터 14일 이내에 회신한다.",
    "제5조 (보존) 접수한 공문의 원본은 5년간 보존한다.",
]

hdoc = HwpxDocument.new()
for line in HWPX_LINES:
    hdoc.add_paragraph(line)
hdoc.save_to_path(str(ROOT / "공용/규정/공문처리지침.hwpx"))
print("공문처리지침.hwpx 생성 완료")

공문처리지침.hwpx 생성 완료


### Step 1-4. 혼합 인코딩(UTF-8·CP949) 파일 생성

이론 4.2절의 인코딩 문제를 재현하기 위해, 구형 시스템에서 결재 로그를 내보낼 때 상단 헤더만 표준 `UTF-8`로 기록하고 본문은 레거시 `CP949` 바이트를 그대로 이어 붙인 **혼합 인코딩 손상 파일**을 생성합니다. 실무 데이터 이관 과정에서 종종 발견되는 유형으로, 단일 인코딩(`UTF-8` 또는 `CP949`)으로 읽으면 파일의 절반이 반드시 깨지므로 품질 지표로 걸러내야 하는 대표적 대상입니다.

In [18]:
LEGACY_HEAD = """# 구 결재 시스템 내보내기 (2019년)

아래는 2019년에 쓰던 구 전자결재 시스템에서 내보낸 결재 기록입니다.
당시 시스템은 한국어 전용 인코딩을 썼고, 내보내기 도구가 머리말만 새 인코딩으로
쓰고 본문은 예전 인코딩 그대로 이어 붙였습니다. 그래서 한 파일 안에 두 인코딩이
섞여 있습니다. 이런 파일은 어느 한 인코딩으로 읽어도 절반이 깨집니다.

"""

LEGACY_BODY = """2019-03-11 총무팀 사무용품 구매 건 승인
2019-03-14 개발1팀 교육비 집행 건 승인
2019-04-02 영업팀 출장 여비 정산 건 반려
"""

(ROOT / "임시/구결재기록_2019.txt").write_bytes(
    LEGACY_HEAD.encode("utf-8") + LEGACY_BODY.encode("euc-kr"))

print("저장소 준비 완료\n")
show_tree(ROOT)

저장소 준비 완료

📁 공용                                       
📁 공용/규정                                 
   공용/규정/공문처리지침.hwpx               7,895 B
   공용/규정/교육비지원규정.pdf             26,874 B
   공용/규정/보안지침_1월.md                   580 B
   공용/규정/보안지침_7월.md                   650 B
   공용/규정/복무규정_2024.md                   625 B
   공용/규정/복무규정_최종.md                  781 B
   공용/규정/복무규정_최종_v2.md               781 B
   공용/규정/출장여비규정.docx              36,974 B
📁 공용/매뉴얼                               
   공용/매뉴얼/비상연락망.md                  419 B
   공용/매뉴얼/시스템매뉴얼.md                  466 B
   공용/부서별예산.xlsx                        5,050 B
📁 영업                                       
📁 영업/계약                                 
   영업/계약/시설이용계약서_스캔본.pdf    30,003 B
   영업/계약/협력사계약현황.md   

## Step 2. 1단계 데이터 목록화(카탈로그 작성)

교재 : 4.4.1 데이터 목록화하기

파이프라인의 첫 단계로 내용을 읽기 전에 **무엇이 어디에 있는지 목록부터 만듭니다.** 교재 4.4.1이 데이터 소스를 데이터베이스 형태로 정리해 나중에 태그를 붙일 수 있게 하라고 한 것이 이 카탈로그입니다.

부서와 접근 등급은 경로에서 유도합니다. 이론 1.4절 Quiz에서 짚은 **버전과 유효 여부를 담을 필드**도 지금 만들어 두고 값은 비워 둡니다.

In [19]:
# 🖥 로컬 작업 : 실무에서는 이 매핑 대신 파일 서버의 ACL이나 협업 도구의 권한을 읽습니다.
#               코랩에는 그런 권한 정보가 없어 폴더 이름으로 대신합니다.
import hashlib
import unicodedata
from pathlib import Path

DEPT_MAP = {
    "공용": ("ALL", "public"),
    "인사": ("HR", "confidential"),
    "영업": ("SALES", "internal"),
    "임시": ("ALL", "draft"),
}


def build_catalog(root: Path):
    rows = []
    for p in sorted(root.rglob("*")):
        if not p.is_file():
            continue
        rel = p.relative_to(root)
        # 💡 [수정] NFD로 저장된 한글을 NFC(표준 완성형)로 정규화합니다.
        top = unicodedata.normalize("NFC", rel.parts[0])
        dept, level = DEPT_MAP.get(top, ("UNKNOWN", "unknown"))
        raw = p.read_bytes()
        rows.append(
            {
                "path": unicodedata.normalize("NFC", rel.as_posix()),
                "name": unicodedata.normalize("NFC", p.name),
                "top": top,
                "ext": p.suffix.lower(),
                "size": len(raw),
                "mtime": p.stat().st_mtime,
                "file_hash": hashlib.sha256(raw).hexdigest()[:16],
                "dept": dept,
                "access_level": level,
                "status": "pending",
                "reject_reason": None,
                "retry_path": None,
                "doc_version": None,
                "is_current": None,
                "valid_from": None,
                "valid_until": None,
                "superseded_by": None,
                "indexed": True,
            }
        )
    return rows


catalog = build_catalog(ROOT)
print(f"카탈로그 항목 {len(catalog)}개\n")
print(f"{'경로':<40} {'형식':>7} {'부서':>7} {'등급':>14} {'크기':>9}")
print("-" * 82)
for c in catalog:
    print(f"{c['path']:<40} {c['ext']:>7} {c['dept']:>7} "
          f"{c['access_level']:>14} {c['size']:>9,}")

카탈로그 항목 18개

경로                                            형식      부서             등급        크기
----------------------------------------------------------------------------------
공용/규정/공문처리지침.hwpx                          .hwpx     ALL         public     7,895
공용/규정/교육비지원규정.pdf                           .pdf     ALL         public    26,874
공용/규정/보안지침_1월.md                             .md     ALL         public       580
공용/규정/보안지침_7월.md                             .md     ALL         public       650
공용/규정/복무규정_2024.md                           .md     ALL         public       625
공용/규정/복무규정_최종.md                             .md     ALL         public       781
공용/규정/복무규정_최종_v2.md                          .md     ALL         public       781
공용/규정/출장여비규정.docx                          .docx     ALL         public    36,974
공용/매뉴얼/비상연락망.md                              .md     ALL         public       419
공용/매뉴얼/시스템매뉴얼.md                             .md     ALL         public       466
공용

In [20]:
import unicodedata

# 첫 번째 파일의 top 출력해보기
test_top = catalog[0]['top']
print("출력:", test_top)
print("글자 수:", len(test_top))  # NFC라면 2, NFD라면 6이 나옵니다.
print("코드의 '공용'과 같은가?:", test_top == "공용")

출력: 공용
글자 수: 2
코드의 '공용'과 같은가?: True


`doc_version`부터 `superseded_by`까지 버전 관리용 다섯 개 필드는 현재 값이 비어 있더라도 **1단계 카탈로그 스키마에 미리 생성해 둡니다.** 이론 1.4절 Quiz에서 살펴본 것처럼 "작년에 폐지된 규정이 왜 계속 검색되느냐"는 사고의 근본 원인은 목록화 단계에 문서의 유효 기간과 폐지 여부를 기록할 필드 자체가 없었기 때문이며 서비스 오픈 이후에 스키마를 뒤늦게 추가하려면 전체 카탈로그와 인덱스를 처음부터 재구축해야 합니다.

또한 `indexed` 필드도 기본값을 `True`로 두고 필터링·중복·권한 정책에 걸릴 때만 `False`로 내리도록 설계하여, 이론 4.6절의 **'물리적 삭제 대신 상태 플래그 표시(Soft Delete)'** 원칙을 코드로 구현했습니다.

카탈로그 단계에서 이미 **인사 폴더 문서가 confidential 등급**으로 보입니다. 이 문서를 전사 챗봇에 인덱싱하면 이론 1.2절에서 말한 권한 우회가 그대로 일어납니다.

`시설이용계약서_스캔본.pdf`는 한 쪽짜리 계약서인데 크기가 다른 PDF·워드 파일과 비슷한 수십 KB라서 **목록만으로는 구별되지 않습니다.** Step 4에서 글자 수와 함께 보면 드러납니다.

## Step 3. 5단계 텍스트 수집과 6단계 인코딩 감지

교재 : 4.4.5 데이터 수집하기, 4.4.6 인코딩 감지하기

이론 1.4절의 열 단계는 **개념의 순서이지 코드의 실행 순서가 아닙니다.** 2번 프라이버시 확인도 3번 필터링도 4번 중복 제거도 전부 문서의 내용을 봐야 판단할 수 있고 내용을 보려면 먼저 텍스트를 추출해야 합니다.

그래서 코드에서는 **5번 수집을 앞으로 당깁니다.** 앞에 둔 것은 Step 2의 카탈로그처럼 파일 크기와 해시같이 내용을 안 봐도 알 수 있는 것뿐입니다.

인코딩부터 다룹니다. 이론 4.2절의 지침은 **추측하고 검증하라**였습니다.

`chardet` 라이브러리가 인코딩을 추정하지만 오판이 잦으므로 개발자가 직접 한글 포함 여부를 검증합니다. 검증 기준은 이론 4.2절에서 본 대로 **한국어 문서라면 읽은 결과에 한글이 있는지 보는 것이 가장 확실하고** 대체 문자 비율보다 잘 걸립니다. 교재 4.4.6절에서도 `chardet` 같은 감지 도구를 사용해 식별한 인코딩을 메타데이터에 기록하도록 안내합니다. 오늘 카탈로그는 인코딩을 남기지 않으며 이 빠진 항목이 청크 점검에서 어떻게 드러나는지는 파이프라인 노트 Step 9에서 봅니다.

In [21]:
import chardet

HANGUL = re.compile(r"[가-힣]")
ENCODING_CANDIDATES = ["utf-8", "cp949", "euc-kr"]


def decode_quality(text):
    """읽은 결과가 얼마나 그럴듯한지 점수를 매깁니다. 한글은 더하고 깨진 글자는 뺍니다"""
    if not text:
        return -1.0
    hangul = len(HANGUL.findall(text)) / len(text)
    broken = text.count("�") / len(text)
    return hangul - broken


def read_text_safe(p: Path, verbose=False):
    """인코딩을 추측하고 검증해서 가장 나은 결과를 돌려줍니다"""
    raw = p.read_bytes()
    if not raw:
        return ""

    guess = chardet.detect(raw)["encoding"]
    tried, best, best_score = [], "", -2.0
    # 1단계 추측을 맨 앞에 두고, 그다음 후보들을 순서대로 테스트합니다
    for enc in [guess] + ENCODING_CANDIDATES:
        if not enc:
            continue
        try:
            text = raw.decode(enc, errors="replace")
        except (LookupError, UnicodeDecodeError):
            continue
        score = decode_quality(text)
        tried.append((enc, round(score, 3)))
        if score > best_score:
            best, best_score = text, score

    if verbose:
        print(f"  chardet 추측 : {guess}")
        print(f"  후보별 점수  : {tried}")
    return best

위 코드에서는 `chardet`의 추정 인코딩을 무조건 채택하지 않고 **후보 인코딩 목록의 첫 번째 순서로만 추가한 뒤, 모든 후보의 디코딩 결과를 동일한 정량 점수(대체 문자 감점 + 한글 음절 비율 가점)로 비교해 최적 인코딩을 선택**했습니다.

만약 1단계(`chardet`)에서 예외가 발생했을 때만 2단계 후보로 넘어가는 단순 `try-except` 폴백으로 구현하면, 1단계가 예외 없이 깨진 문자열을 반환하는 조용한 실패 앞에서는 폴백이 전혀 작동하지 않습니다. 이제 앞서 만든 혼합 인코딩 손상 파일로 검증 동작을 확인합니다.

In [22]:
# 1단계 추측이 실패를 내는지 직접 확인합니다
broken = ROOT / "임시/구결재기록_2019.txt"
text = read_text_safe(broken, verbose=True)
print(f"\n채택한 결과 : {len(text)}자, "
      f"한글 {len(HANGUL.findall(text))/len(text):.0%}, "
      f"대체 문자 {text.count(chr(0xFFFD))/len(text):.0%}")
print(repr(text[:60]))
print("...")
print(repr(text[-60:]))

  chardet 추측 : cp720
  후보별 점수  : [('cp720', 0.0), ('utf-8', 0.231), ('cp949', 0.111), ('euc-kr', -0.406)]

채택한 결과 : 316자, 한글 42%, 대체 문자 19%
'# 구 결재 시스템 내보내기 (2019년)\n\n아래는 2019년에 쓰던 구 전자결재 시스템에서 내보낸 결재 기'
...
'������ ���� �� ����\n2019-04-02 ������ ���� ���� ���� �� �ݷ�\n'


`chardet`이 엉뚱한 인코딩을 내놓았는데 **예외는 나지 않았습니다.** 그대로 믿었다면 한글이 하나도 없는 글자 뭉치를 본문으로 삼았을 것입니다.

검증을 붙이니 후보 중 점수가 가장 높은 것이 뽑혔습니다. 앞부분 한글은 살아 있지만 뒷부분에 대체 문자가 몰려 **여전히 절반은 깨져 있습니다.** 살릴 수 없는 파일이라 Step 6의 필터가 사유와 함께 걸러 냅니다.

이제 형식별 파서를 붙입니다. 이론 1.2절은 **텍스트를 추출하지 못하면 중복이 있는지도 개인정보가 있는지도 알 수 없어서** 형식 난립을 다섯 문제 중 첫 번째로 꼽았습니다.

In [23]:
def parse_text(p):
    return read_text_safe(p)


def parse_docx(p):
    from docx import Document
    d = Document(p)
    parts = [para.text for para in d.paragraphs if para.text.strip()]
    for table in d.tables:
        for row in table.rows:
            parts.append(" | ".join(cell.text for cell in row.cells))
    return "\n".join(parts)


def parse_xlsx(p):
    import openpyxl
    wb = openpyxl.load_workbook(p, data_only=True)
    parts = []
    for ws in wb.worksheets:
        parts.append(f"[시트: {ws.title}]")
        for row in ws.iter_rows(values_only=True):
            cells = [str(c) for c in row if c is not None]
            if cells:
                parts.append(" | ".join(cells))
    return "\n".join(parts)


def parse_pdf(p):
    from pypdf import PdfReader
    reader = PdfReader(p)
    pages = []
    for page in reader.pages:
        t = page.extract_text() or ""
        pages.append(t.replace("\xa0", " "))    # PDF에서 자주 섞여 나오는 특수 공백
    return "\n".join(pages).strip()


def parse_hwpx(p):
    from hwpx.document import HwpxDocument
    return HwpxDocument.open(str(p)).text.plain()


PARSERS = {".md": parse_text, ".txt": parse_text,
           ".docx": parse_docx, ".xlsx": parse_xlsx,
           ".pdf": parse_pdf, ".hwpx": parse_hwpx}

print("붙인 파서 :", ", ".join(sorted(PARSERS)))

붙인 파서 : .docx, .hwpx, .md, .pdf, .txt, .xlsx


`parse_pdf`에서는 PDF 줄바꿈 방지용 특수 공백(` `, Non-breaking Space)을 일반 공백으로 정규화했습니다(이론 4.1절). 육안으로는 일반 공백과 구분되지 않지만 바이트 값이 달라서, 정규화하지 않으면 동일 문서의 사본 중 한쪽에만 특수 공백이 섞여 있을 때 해시값이 달라져 완전 중복 판정을 빠져나가게 됩니다.

또한 `parse_hwpx`에서 사용한 `doc.text.plain()`은 `python-hwpx` 6.0부터 도입된 메서드명이며 구형 `export_text()`는 7.0에서 제거될 예정이므로 Step 0에서처럼 파서 버전을 고정해 운영합니다.

In [24]:
# 파싱 실행. 실패도 유형을 나눠 기록합니다
for c in catalog:
    fn = PARSERS.get(c["ext"])
    if fn is None:
        c["status"] = "unsupported"
        c["reject_reason"] = f"지원하지 않는 형식: {c['ext']}"
        c["retry_path"] = None                 # 파서를 새로 붙이기 전에는 재시도해도 같습니다
        c["text"] = ""
        continue
    try:
        c["text"] = fn(ROOT / c["path"])
        c["status"] = "parsed"
    except Exception as e:
        c["status"] = "parse_failed"
        c["reject_reason"] = f"{type(e).__name__}: {e}"
        c["retry_path"] = "다른 파서"           # 같은 파일을 다른 도구로 다시 실행할 대상
        c["text"] = ""

print(Counter(c["status"] for c in catalog))
for c in catalog:
    if c["status"] != "parsed":
        print(f"  [{c['status']}] {c['path']} : {c['reject_reason']}")

Counter({'parsed': 18})


`retry_path`를 함께 기록해 **재시도할 실패와 아닌 실패를 구분했습니다.** 이론 4.3절에서 추출에 실패한 문서는 버리지 않고 다른 경로로 분류한다고 한 내용입니다.

지원하지 않는 형식은 몇 번을 다시 실행해도 같으니 파서를 새로 붙여야 하고 파싱 중에 예외가 난 파일은 **다른 도구로 다시 실행하면 될 수도 있습니다.** 두 가지를 한 필드에 `실패`로만 적어 두면 나중에 무엇을 다시 실행해야 하는지 알 수 없습니다.

## Step 4. 문서별 추출 품질 지표 산출

이론 2.7절의 표대로 **모든 실패가 예외를 내지는 않습니다.** 방금 파싱에서 `parsed`가 나왔다고 글자가 제대로 나왔다는 보장은 없습니다.

그래서 이론 6.1절의 지표들로 감지 장치를 직접 만듭니다.

In [25]:
def doc_metrics(c):
    """이론 6.1절의 지표들. 전부 자동으로 계산되고 계산이 쌉니다"""
    t = c.get("text", "")
    n = len(t)
    lines = [l for l in t.split("\n") if l.strip()]
    return {
        "char_len": n,
        "chars_per_kb": round(n / max(c["size"] / 1024, 0.001), 1),
        "hangul_ratio": round(len(HANGUL.findall(t)) / n, 3) if n else 0.0,
        "replacement_ratio": round(t.count("�") / n, 3) if n else 0.0,
        "avg_line_len": round(sum(len(l) for l in lines) / len(lines), 1) if lines else 0.0,
    }


for c in catalog:
    c.update(doc_metrics(c))

print(f"{'경로':<40} {'글자':>6} {'KB당':>7} {'한글':>6} {'깨짐':>6} {'줄길이':>7}")
print("-" * 78)
for c in catalog:
    print(f"{c['path']:<40} {c['char_len']:>6} {c['chars_per_kb']:>7} "
          f"{c['hangul_ratio']:>6.0%} {c['replacement_ratio']:>6.0%} {c['avg_line_len']:>7}")

경로                                           글자     KB당     한글     깨짐     줄길이
------------------------------------------------------------------------------
공용/규정/공문처리지침.hwpx                           197    25.6    64%     0%    32.0
공용/규정/교육비지원규정.pdf                           306    11.7    62%     0%    26.9
공용/규정/보안지침_1월.md                            256   452.0    63%     0%    30.6
공용/규정/보안지침_7월.md                            290   456.9    62%     0%    27.6
공용/규정/복무규정_2024.md                          289   473.5    58%     0%    24.8
공용/규정/복무규정_최종.md                            355   465.5    60%     0%    30.8
공용/규정/복무규정_최종_v2.md                         355   465.5    60%     0%    30.8
공용/규정/출장여비규정.docx                           228     6.3    64%     0%    37.2
공용/매뉴얼/비상연락망.md                             269   657.4    28%     0%    43.3
공용/매뉴얼/시스템매뉴얼.md                            216   474.6    58%     0%    29.4
공용/부서별예산.xlsx                               102    20.7    31% 

출력된 품질 지표 표에서 세 가지 이상 징후가 즉시 포착됩니다.

- **스캔본 PDF(`시설이용계약서_스캔본.pdf`)** : 파일 크기는 수십 KB인데 추출 글자 수와 KB당 글자 수가 모두 `0`입니다
- **빈 문서(`신규문서.md`)** : 추출 글자 수와 파일 크기가 모두 `0`입니다
- **혼합 인코딩 문서(`구결재기록_2019.txt`)** : 대체 문자(``) 비율이 비정상적으로 높습니다

특히 **스캔본과 빈 문서는 둘 다 추출 글자 수가 0자이지만 처리 경로가 완전히 달라야 합니다.** 파일 크기까지 0B인 빈 문서는 인덱싱에서 제외(`filtered`)하고, 파일 크기는 큰데 글자만 0자인 스캔본은 폐기하지 않고 OCR·VLM 재처리 경로(`needs_ocr`)로 분류합니다(이론 6.1절).

다만 이 지표들을 그대로 임계값으로 쓰면 안 되는 이유도 표에 함께 나와 있습니다.

**KB당 글자 수는 형식마다 기준이 다릅니다.** docx와 xlsx는 압축 아카이브라 글자가 멀쩡히 나와도 KB당 글자 수가 낮게 나오므로 하나의 임계값으로 모든 형식을 자르면 이 둘이 통째로 걸립니다. **형식별로 따로 보거나 0자인 경우에만 씁니다.**

**한글 비율도 마찬가지입니다.** 비상연락망은 연락처와 이메일 주소가 많아 한글 비율이 가장 낮은 축인데 멀쩡한 문서이고 부서별 예산도 숫자가 많아 낮습니다. 이론 6.1절이 경고한 대로 **"한글 비율이 낮으면 버린다"는 규칙은 영문 기술 문서와 연락처 목록과 표를 버립니다.**

## Step 5. 2단계 프라이버시 및 규정 준수 확인

교재 : 4.4.2 프라이버시 및 법적 규정 준수 확인하기

민감 정보를 찾습니다. 이론 5.3절에서 본 대로 **정규식은 1차 탐지 수단**이라 형식이 정해진 식별자는 낮은 연산 비용으로 빠르게 검출하지만 이름과 문맥은 못 잡습니다.

먼저 흔히 쓰는 패턴을 그대로 써 보고 무엇이 걸리는지 셉니다.

In [26]:
# 이론 5.3절이 예로 든 그 규칙입니다. 계좌번호를 "숫자-숫자-숫자"로 적었습니다
PII_PATTERNS_NAIVE = {
    "주민등록번호": r"\b\d{6}\s?-\s?[1-4]\d{6}\b",
    "휴대전화": r"\b01[016-9]\s?-?\s?\d{3,4}\s?-?\s?\d{4}\b",
    "계좌번호": r"\b\d{2,3}-\d{2,6}-\d{4,8}\b",
    "이메일": r"\b[\w.+-]+@[\w-]+\.[\w.]+\b",
    "카드번호": r"\b\d{4}-\d{4}-\d{4}-\d{4}\b",
}


def scan_pii(text, patterns):
    found = {}
    for name, pat in patterns.items():
        hits = re.findall(pat, text)
        if hits:
            found[name] = hits
    return found


def pattern_report(catalog, patterns, title):
    """패턴마다 몇 건이 걸렸는지 세고 표본을 보여 줍니다 (이론 5.3절)"""
    per_pattern, samples = Counter(), {}
    for c in catalog:
        if c["status"] != "parsed":
            continue
        for name, hits in scan_pii(c["text"], patterns).items():
            per_pattern[name] += len(hits)
            samples.setdefault(name, []).extend(hits[:2])
    print(f"[{title}]")
    for name, n in per_pattern.most_common():
        print(f"  {name:<10} {n:>3}건   표본 {samples[name][:3]}")
    return per_pattern


naive_counts = pattern_report(catalog, PII_PATTERNS_NAIVE, "흔히 쓰는 패턴")

[흔히 쓰는 패턴]
  계좌번호         8건   표본 ['010-2345-6789', '010-3456-7890', '010-2345-6789']
  휴대전화         6건   표본 ['010-2345-6789', '010-3456-7890', '010-2345-6789']
  이메일          3건   표본 ['infra-oncall@ipaltech.example.com', 'dev1-lead@ipaltech.example.com']
  주민등록번호       3건   표본 ['880315-1234567', '920708-2345678']


출력된 계좌번호 적발 표본을 보면 **일반 휴대전화 번호(`010-2345-6789`)가 계좌번호로 잘못 잡혀 있습니다.** 이론 5.3절에서 살펴본 것처럼 `'숫자 묶음-숫자 묶음-숫자 묶음'`이라는 단순 계좌번호 패턴에 휴대전화 번호가 그대로 부합하기 때문입니다.

잠시 후 비식별화 정책을 적용할 때 **계좌번호 포함 문서는 전체 제외(`exclude`)하고 연락처만 있는 문서는 마스킹(`mask`) 후 인덱싱**하기로 구분할 텐데, 이 오탐을 방치하면 마스킹해서 정상 활용할 수 있는 `비상연락망.txt`까지 고위험 문서로 분류되어 통째로 인덱싱에서 빠지게 됩니다. 따라서 계좌번호 정규식 앞에 `'휴대전화 앞자리(`010` 등)로 시작하지 않을 것'`이라는 부정형 전방 탐색 조건을 추가해 수정합니다.

In [27]:
PII_PATTERNS = dict(PII_PATTERNS_NAIVE)
PII_PATTERNS["계좌번호"] = r"\b(?!01[016-9]-)\d{2,3}-\d{2,6}-\d{4,8}\b"

fixed_counts = pattern_report(catalog, PII_PATTERNS, "고친 패턴")

print(f"\n계좌번호 : {naive_counts['계좌번호']}건 -> {fixed_counts['계좌번호']}건")
print("줄어든 만큼이 연락처를 계좌번호로 잘못 잡고 있던 건수입니다")

[고친 패턴]
  휴대전화         6건   표본 ['010-2345-6789', '010-3456-7890', '010-2345-6789']
  이메일          3건   표본 ['infra-oncall@ipaltech.example.com', 'dev1-lead@ipaltech.example.com']
  주민등록번호       3건   표본 ['880315-1234567', '920708-2345678']
  계좌번호         2건   표본 ['110-234-567890', '123-45-6789012']

계좌번호 : 8건 -> 2건
줄어든 만큼이 연락처를 계좌번호로 잘못 잡고 있던 건수입니다


`(?!01[016-9]-)`가 "바로 뒤가 휴대전화 앞자리가 아닐 것"을 뜻합니다. 이런 조건을 앞에 붙여 특정 패턴을 빼는 방식은 정규식에서 자주 쓰는 손질입니다.

**고친 뒤에도 이 패턴이 완벽하지는 않습니다.** 사업자등록번호나 날짜 표기 중에도 걸릴 형식이 있으므로 방금처럼 **패턴마다 몇 건이 걸렸는지 세고 걸린 것의 표본을 열어 보는 습관**이 중요합니다.

In [28]:
# 문서별로 무엇이 걸렸는지 카탈로그에 기록합니다
for c in catalog:
    if c["status"] != "parsed":
        continue
    hits = scan_pii(c["text"], PII_PATTERNS)
    c["pii"] = {name: len(v) for name, v in hits.items()}
    if c["pii"]:
        print(f"[{c['access_level']:>12}] {c['path']}")
        print(f"               {c['pii']}")

[      public] 공용/매뉴얼/비상연락망.md
               {'휴대전화': 3, '이메일': 3}
[confidential] 인사/평가/2026상반기_인사평가.md
               {'주민등록번호': 3, '휴대전화': 3, '계좌번호': 2}


인사평가 문서에서 주민등록번호와 연락처와 계좌번호가, 비상연락망에서 연락처와 이메일이 잡혔습니다.

**협력사 계약 문서에서는 아무것도 안 잡혔는데** 그 안에는 담당자 이름 세 개와 계약 금액이 그대로 있습니다. 이론 5.3절의 정규식 한계대로 형식이 있는 것은 잡고 **이름과 문맥상 민감한 정보는 못 잡습니다.**

개체명 인식 기반 탐지와 권한 설계는 보안 주간에서 다룹니다.

In [31]:
# 실습 : 문서마다 처리 방침을 결정하는 함수를 완성하세요
# 규칙 (이론 5.2절의 세 기준을 순서대로 적용한 것입니다)
#  1) access_level 이 confidential 이면 제외("exclude"). 사유에 "권한 분리 후 재검토"를 남길 것
#  2) 주민등록번호, 계좌번호, 카드번호(HIGH_RISK)가 있으면 제외("exclude")
#  3) 휴대전화나 이메일 등 일반 개인정보만 있으면 마스킹("mask") 후 포함
#  4) 나머지는 포함("include")

HIGH_RISK = {"주민등록번호", "계좌번호", "카드번호"}


def decide_policy(c):
    if c["access_level"] == "confidential":
        return "exclude", "기밀 등급 문서. 권한 분리 후 재검토"

    pii = c.get("pii", {})
    high = HIGH_RISK & set(pii)
    if high:
        return "exclude", f"고위험 식별자 포함: {', '.join(sorted(high))}"

    if pii:
        return "mask", f"마스킹 대상: {', '.join(sorted(pii))}"

    return "include", "특이사항 없음"

<details class="lms-answer">
<summary>풀이 보기</summary>

**빈칸 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 1) 기밀 등급 제외
if c["access_level"] == "confidential":
    return "exclude", "기밀 등급 문서. 권한 분리 후 재검토"

# 2) 고위험 식별자 교집합 추출
high = HIGH_RISK & set(pii)

# 3) 일반 개인정보 마스킹
if pii:
    return "mask", f"마스킹 대상: {', '.join(sorted(pii))}"
```

<details>
<summary>전체 완성 코드 보기 (복사·붙여넣기용)</summary>

```python
def decide_policy(c):
    if c["access_level"] == "confidential":
        return "exclude", "기밀 등급 문서. 권한 분리 후 재검토"

    pii = c.get("pii", {})
    high = HIGH_RISK & set(pii)
    if high:
        return "exclude", f"고위험 식별자 포함: {', '.join(sorted(high))}"

    if pii:
        return "mask", f"마스킹 대상: {', '.join(sorted(pii))}"

    return "include", "특이사항 없음"
```

</details>

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

**정답 코드에 들어 있는 판단**

**등급을 가장 먼저 봅니다.** 지금 우리 시스템에는 권한 기반 검색이 없으므로 개인정보가 하나도 없어도 기밀 문서는 제외하고 사유에 "권한 분리 후 재검토"를 남겼습니다. 이 한 줄이 있어야 나중에 권한 필터를 붙였을 때 **무엇을 다시 포함해야 하는지** 알 수 있습니다.

**위험도에 따라 처리를 나눕니다.** 주민번호와 계좌번호는 제외, 연락처와 이메일은 마스킹입니다. 모든 개인정보를 같게 취급하면 쓸 수 있는 문서가 너무 줄어듭니다.

비상연락망이 `mask`로 나왔는지 확인합니다. 조금 전에 계좌번호 패턴을 고치지 않았다면 이 문서는 `exclude`가 되어 **가장 자주 찾는 종류의 문서가 오탐 때문에 통째로 사라졌을 것입니다.**

</details>

In [32]:
if decide_policy(catalog[0]) is None:
    print("⚠ decide_policy 가 아직 비어 있습니다. 위 실습 셀을 완성한 뒤 다시 실행하세요.")
else:
    for c in catalog:
        if c["status"] != "parsed":
            continue
        action, reason = decide_policy(c)
        c["policy"] = action
        c["policy_reason"] = reason
        if action == "exclude":
            c["indexed"] = False        # 판정을 실제로 집행하는 줄
            c["reject_reason"] = reason
        print(f"{action:>8}  {c['path']:<40} {reason}")

 include  공용/규정/공문처리지침.hwpx                        특이사항 없음
 include  공용/규정/교육비지원규정.pdf                        특이사항 없음
 include  공용/규정/보안지침_1월.md                         특이사항 없음
 include  공용/규정/보안지침_7월.md                         특이사항 없음
 include  공용/규정/복무규정_2024.md                       특이사항 없음
 include  공용/규정/복무규정_최종.md                         특이사항 없음
 include  공용/규정/복무규정_최종_v2.md                      특이사항 없음
 include  공용/규정/출장여비규정.docx                        특이사항 없음
    mask  공용/매뉴얼/비상연락망.md                          마스킹 대상: 이메일, 휴대전화
 include  공용/매뉴얼/시스템매뉴얼.md                         특이사항 없음
 include  공용/부서별예산.xlsx                            특이사항 없음
 include  영업/계약/시설이용계약서_스캔본.pdf                    특이사항 없음
 include  영업/계약/협력사계약현황.md                         특이사항 없음
 exclude  인사/평가/2026상반기_인사평가.md                    기밀 등급 문서. 권한 분리 후 재검토
 include  임시/구결재기록_2019.txt                        특이사항 없음
 include  임시/시스템매뉴얼_복사.md                          특이사항 없음
 include  임시/신규문서.md            

위 코드에서 `action == "exclude"`일 때 `c["indexed"] = False`로 내리는 줄이 핵심입니다. **정책 판정을 기록하는 일과 실제 인덱싱 대상에서 제외하는 집행은 별개의 단계이기 때문입니다.**

만약 `c["indexed"] = False` 한 줄을 누락하면 카탈로그의 `policy` 열에는 `"exclude"`라고 정상 출력되지만 뒷단 스냅샷 저장 코드는 `indexed == True` 여부만 확인하므로 **주민등록번호가 포함된 기밀 인사평가 문서가 최종 검색 코퍼스에 그대로 적재됩니다.** 파이썬 예외도 전혀 발생하지 않으므로 판정 결과가 실제 데이터 흐름에 집행되는지 반드시 확인해야 합니다.

In [33]:
def mask_pii(text):
    out = text
    for name, pat in PII_PATTERNS.items():
        out = re.sub(pat, f"[{name} 마스킹]", out)
    return out


sample = next(c for c in catalog if c["path"].endswith("비상연락망.md"))
print(mask_pii(sample["text"]))

# 야간 휴일 비상 연락망

시스템 장애가 발생하면 아래 순서로 연락합니다.

1. 1차 인프라팀 당직 - [휴대전화 마스킹], [이메일 마스킹]
2. 2차 개발1팀 팀장 - [휴대전화 마스킹], [이메일 마스킹]
3. 3차 정보보호팀 - [휴대전화 마스킹], [이메일 마스킹]

연락이 닿지 않으면 사내 포털의 장애 신고 메뉴에 기록을 남깁니다.



**마스킹은 원본 파일이 아니라 읽어 온 텍스트에만 적용했습니다.** 이론 5.4절에서 본 대로 원본은 그대로 두고 인덱싱용 사본에만 마스킹했고 4.6절의 "삭제 대신 표시"와 같은 원칙입니다. 원본까지 고치면 되돌릴 수 없고 나중에 규칙이 바뀌었을 때 다시 처리할 수가 없습니다.

## Step 6. 3단계 문서 단위 품질 필터링

교재 : 4.4.3 데이터 필터링하기

이론 4.3절이 버리기 전에 세라고 했으니 기준별로 몇 건이 걸리는지 반드시 기록합니다.

이론 6.1절의 구분도 넣습니다. **제외하는 것과 다른 경로로 보내는 것은 다르고** 스캔본은 제외할 문서가 아니라 OCR 경로로 보낼 문서입니다.

In [34]:
# 실습 : 필터 규칙의 핵심 조건을 채워 각 규칙이 걸러낸 건수를 집계하세요
# 규칙 (순서가 중요합니다. 위에서부터 걸리면 아래는 보지 않습니다)
#  1) 글자가 0자(c["char_len"] == 0)인데 파일 크기(c["size"])가 MIN_SCAN_BYTES 초과 -> status "needs_ocr"
#  2) 공백 제외 글자 수(len(text.strip()))가 MIN_CHARS 미만 -> status "filtered", stats["too_short"]
#  3) 상투적 문구(보일러플레이트) 비율이 MAX_BOILERPLATE 초과 -> status "filtered", stats["boilerplate"]
#  4) 대체 문자 비율이 MAX_REPLACEMENT 초과 -> status "filtered", stats["encoding_broken"]

MIN_CHARS = 50               # 이보다 짧으면 본문이 없다고 봅니다
MIN_SCAN_BYTES = 10_000      # 이보다 큰 파일이 0자면 그림만 든 문서로 봅니다
MAX_REPLACEMENT = 0.05
MAX_BOILERPLATE = 0.30


def apply_filters(catalog):
    stats = Counter()
    for c in catalog:
        if c["status"] != "parsed":
            continue
        text = c["text"]

        ### START CODE HERE ### (규칙 1과 규칙 2의 조건식 2줄만 채우세요)
        # 1) 파일은 큰데 글자가 없음 -> 스캔본 의심 (버리지 않고 OCR 경로로 분리)
        if c["char_len"] == 0 and c["size"] > MIN_SCAN_BYTES:
            c["status"] = "needs_ocr"
            c["reject_reason"] = f"글자 0자, 파일 {c['size']:,}B. 스캔본 의심"
            c["retry_path"] = "OCR"
            c["indexed"] = False
            stats["needs_ocr"] += 1
            continue

        # 2) 본문이 너무 짧음 -> 공백 제거 본문 길이가 MIN_CHARS 미만인지 검사
        if len(text.strip()) < MIN_CHARS:
            c["status"] = "filtered"
            c["reject_reason"] = f"본문 부족 ({len(text.strip())}자)"
            c["indexed"] = False
            stats["too_short"] += 1
            continue
        ### END CODE HERE ###
        

        # 3) 상투적 문구(보일러플레이트) 비율 (미리 작성됨)
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        boiler = sum(1 for l in lines if l.endswith(":") or l.endswith("："))
        if lines and boiler / len(lines) > MAX_BOILERPLATE:
            c["status"] = "filtered"
            c["reject_reason"] = f"상투적 문구(보일러플레이트) 비율 {boiler/len(lines):.0%}"
            c["indexed"] = False
            stats["boilerplate"] += 1
            continue

        # 4) 인코딩이 살아나지 않음 (미리 작성됨)
        if c["replacement_ratio"] > MAX_REPLACEMENT:
            c["status"] = "filtered"
            c["reject_reason"] = f"대체 문자 비율 {c['replacement_ratio']:.0%}"
            c["indexed"] = False
            stats["encoding_broken"] += 1
            continue

        stats["passed"] += 1
    return stats

<details class="lms-answer">
<summary>풀이 보기</summary>

**빈칸 정답 코드 (`### START CODE HERE ###` 영역)**

```python
is_scan_suspect = (c["char_len"] == 0 and c["size"] > MIN_SCAN_BYTES)
# ...
is_too_short = (len(text.strip()) < MIN_CHARS)
```

<details>
<summary>전체 완성 코드 보기 (복사·붙여넣기용)</summary>

```python
def apply_filters(catalog):
    stats = Counter()
    for c in catalog:
        if c["status"] != "parsed":
            continue
        text = c["text"]

        # 1) 파일은 큰데 글자가 없음 -> 스캔본 의심. 버리지 않고 다른 경로로
        if c["char_len"] == 0 and c["size"] > MIN_SCAN_BYTES:
            c["status"] = "needs_ocr"
            c["reject_reason"] = f"글자 0자, 파일 {c['size']:,}B. 스캔본 의심"
            c["retry_path"] = "OCR"
            c["indexed"] = False
            stats["needs_ocr"] += 1
            continue

        # 2) 본문이 너무 짧음
        if len(text.strip()) < MIN_CHARS:
            c["status"] = "filtered"
            c["reject_reason"] = f"본문 부족 ({len(text.strip())}자)"
            c["indexed"] = False
            stats["too_short"] += 1
            continue

        # 3) 상투적 문구(보일러플레이트) 비율
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        boiler = sum(1 for l in lines if l.endswith(":") or l.endswith("："))
        if lines and boiler / len(lines) > MAX_BOILERPLATE:
            c["status"] = "filtered"
            c["reject_reason"] = f"상투적 문구(보일러플레이트) 비율 {boiler/len(lines):.0%}"
            c["indexed"] = False
            stats["boilerplate"] += 1
            continue

        # 4) 인코딩이 살아나지 않음
        if c["replacement_ratio"] > MAX_REPLACEMENT:
            c["status"] = "filtered"
            c["reject_reason"] = f"대체 문자 비율 {c['replacement_ratio']:.0%}"
            c["indexed"] = False
            stats["encoding_broken"] += 1
            continue

        stats["passed"] += 1
    return stats
```

</details>

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

규칙마다 잡는 문서가 다릅니다.

<div style="font-size: 18px; line-height: 1.6;">

| 규칙 | 걸리는 문서 | 처리 |
|---|---|---|
| 스캔본 의심 | 시설이용계약서_스캔본.pdf | OCR 경로로 |
| 본문 부족 | 임시/신규문서.md | 제외 표시 |
| 상투적 문구(보일러플레이트) 비율 | 임시/템플릿.md | 제외 표시 |
| 대체 문자 | 임시/구결재기록_2019.txt | 제외 표시 |

</div>

**순서도 중요합니다.** 스캔본 판정을 맨 앞에 두지 않으면 글자 0자인 스캔본이 본문 부족 규칙에 걸려 제외되고 사유만 보면 빈 문서와 구별이 안 됩니다. **같은 0자라도 하나는 제외하고 하나는 다시 처리하므로** 둘을 갈라 주는 규칙이 먼저 와야 합니다.

**한글 비율은 필터에 넣지 않았습니다.** 이론 6.1절이 경고한 대로 "한글 비율이 낮으면 버린다"는 규칙은 **영문 기술 문서와 숫자 위주의 엑셀을 버립니다.** 한글 비율은 인코딩을 고를 때 판단 근거로 쓰고 문서를 버리는 기준으로는 쓰지 않았습니다.

**사유는 반드시 기록합니다.** `reject_reason`이 없으면 나중에 "왜 이 문서가 검색되지 않나요"라는 문의에 답할 수 없습니다.

</details>

In [35]:
stats = apply_filters(catalog)
if not stats:
    print("⚠ apply_filters 가 아직 비어 있습니다. 위 실습 셀을 완성한 뒤 다시 실행하세요.\n")

print("필터 통계 :", dict(stats))
print(Counter(c["status"] for c in catalog))
print()
for c in catalog:
    if c["status"] in ("filtered", "needs_ocr"):
        print(f"  [{c['status']:<10}] {c['path']:<40} {c['reject_reason']}")

필터 통계 : {'passed': 14, 'needs_ocr': 1, 'encoding_broken': 1, 'too_short': 1, 'boilerplate': 1}
Counter({'parsed': 14, 'filtered': 3, 'needs_ocr': 1})

  [needs_ocr ] 영업/계약/시설이용계약서_스캔본.pdf                    글자 0자, 파일 30,003B. 스캔본 의심
  [filtered  ] 임시/구결재기록_2019.txt                        대체 문자 비율 19%
  [filtered  ] 임시/신규문서.md                               본문 부족 (0자)
  [filtered  ] 임시/템플릿.md                                상투적 문구(보일러플레이트) 비율 45%


### Step 6-1. 줄 단위 정제 규칙 적용
교재 : 4.4.3 데이터 필터링하기, 4.4.4 데이터 중복 제거하기

Step 6은 문서 단위로 걸렀습니다. 교재 4.4.3의 '데이터 정제 핵심 규칙'에는 줄과 문장 단위로 걸 수 있는 규칙이 여럿 있습니다. 중괄호가 든 문장(대개 플레이스홀더), HTML·JavaScript 흔적, 지나치게 짧은 문장, 내비게이션 바와 버튼 문구, 원치 않는 단어가 든 텍스트를 거르는 규칙이고 교재 4.4.4는 연속해서 반복되는 문장은 첫 번째만 남기는 방법도 소개합니다.

우리 저장소에는 웹 노이즈가 거의 없으므로 사내 위키에서 내보낸 듯한 표본 글을 셀 안에 함께 넣습니다. 이론 4.3절과 4.6절의 원칙대로 **줄을 지우지 않고 어느 줄이 어느 규칙에 걸리는지 세고 표시만** 남깁니다.

In [36]:
# 교재 4.4.3 '데이터 정제 핵심 규칙'을 줄 단위로 걸어 봅니다. 줄을 지우지 않고 표시만 남깁니다
WEB_EXPORT = """홈 > 공지사항 > 인사
<div class="notice-body">
{{user_name}}님, 아래 공지를 확인하세요.
2026년 하반기 교육 신청은 사내 포털의 교육 신청 메뉴에서 받습니다.
2026년 하반기 교육 신청은 사내 포털의 교육 신청 메뉴에서 받습니다.
<script>trackView('notice-1182');</script>
대외비
Lorem ipsum dolor sit amet
좋아요 3 | 공유하기 | 목록으로"""

MIN_LINE_CHARS = 10                          # '#' 을 뗀 뒤 이보다 짧으면 '너무 짧음'
UNWANTED_WORDS = ("lorem ipsum", "대외비")    # 더미 문구와 워터마크 단어
LINE_RULES = [                               # 위에서부터 처음 걸린 규칙 하나만 적습니다
    ("중괄호", lambda l: re.search(r"\{[^{}]*\}", l)),
    ("HTML·JS", lambda l: re.search(r"</?[a-zA-Z][^>]*>", l)),
    ("내비게이션·버튼", lambda l: re.fullmatch(r"\S+( > \S+)+", l) or re.search(r"좋아요|공유하기|목록으로", l)),
    ("원치 않는 단어", lambda l: any(w in l.lower() for w in UNWANTED_WORDS)),
    ("너무 짧음", lambda l: len(l.lstrip("#").strip()) < MIN_LINE_CHARS),
]


def mark_lines(text):
    """[(줄 번호, 규칙, 줄)] 을 돌려줍니다. 연속 중복은 첫 줄을 남기고 뒤 줄을 표시합니다"""
    marks, prev = [], None
    for n, l in enumerate((x.strip() for x in text.split("\n")), 1):
        if l:
            rule = "연속 중복" if l == prev else next((r for r, hit in LINE_RULES if hit(l)), None)
            if rule:
                marks.append((n, rule, l))
            prev = l
    return marks


line_marks = {c["path"]: mark_lines(c["text"]) for c in catalog if c["status"] == "parsed"}
line_marks["(웹 내보내기 표본)"] = mark_lines(WEB_EXPORT)
all_marks = [(p, n, r, l) for p, ms in line_marks.items() for n, r, l in ms]

for rule, cnt in Counter(r for _, _, r, _ in all_marks).most_common():
    p, n, _, l = next(m for m in all_marks if m[2] == rule)      # 규칙마다 첫 예 하나
    print(f"{rule:<8} {cnt:>3}줄   예) {p.split('/')[-1]} {n}행 '{l[:30]}'")

short = [l for _, _, r, l in all_marks if r == "너무 짧음"]
heads = [l for l in short if l.startswith("#") or re.match(r"제\d+조", l)]
print(f"\n너무 짧음 {len(short)}줄 중 제목·조항 줄 {len(heads)}줄")

너무 짧음     16줄   예) 교육비지원규정.pdf 2행 '제1조 (목적)'
원치 않는 단어   3줄   예) 2026상반기_인사평가.md 1행 '# 2026년 상반기 인사평가 결과 (대외비)'
내비게이션·버튼   2줄   예) (웹 내보내기 표본) 1행 '홈 > 공지사항 > 인사'
HTML·JS    2줄   예) (웹 내보내기 표본) 2행 '<div class="notice-body">'
중괄호        1줄   예) (웹 내보내기 표본) 3행 '{{user_name}}님, 아래 공지를 확인하세요.'
연속 중복      1줄   예) (웹 내보내기 표본) 5행 '2026년 하반기 교육 신청은 사내 포털의 교육 신청 '

너무 짧음 16줄 중 제목·조항 줄 15줄


웹 노이즈 규칙(중괄호, HTML·JS, 내비게이션·버튼, 연속 중복)은 표본 글에서만 걸렸고 파서로 추출한 사내 문서에는 이런 흔적이 거의 없습니다.

**"너무 짧음"에 걸린 줄은 거의 모두 문서 제목과 조항 제목입니다.** 그대로 지우면 이론 1.1절에서 본 구조 기반 청킹에 쓸 제목이 사라지므로 사내 규정에 쓰려면 제목 줄을 예외로 둬야 합니다.

원치 않는 단어 목록에는 워터마크 단어 "대외비"(이론 2.6절)를 일부러 넣었고 표본의 워터마크 줄과 함께 인사평가 문서의 정상 제목까지 걸렸습니다. 단어 목록이 필요한 문장까지 잡는 것은 이론 4.7절에서 본 유해성 필터의 부작용과 같은 종류의 문제입니다.

## Step 7. 4단계 완전·근사 중복 검출

교재 : 4.4.4 데이터 중복 제거하기

이론 4.4절의 두 층위를 **연산 비용이 낮은 방법부터** 순서대로 처리합니다.

In [37]:
# 완전 중복 : 텍스트 해시로 즉시 판별
alive = [c for c in catalog if c["status"] == "parsed"]

by_hash = {}
for c in alive:
    h = hashlib.sha256(c["text"].encode()).hexdigest()
    c["text_hash"] = h
    by_hash.setdefault(h, []).append(c)

exact_groups = {h: v for h, v in by_hash.items() if len(v) > 1}
print(f"완전 중복 그룹 {len(exact_groups)}개")
for h, group in exact_groups.items():
    print(f"  그룹 ({len(group)}건) : {[c['path'] for c in group]}")

완전 중복 그룹 2개
  그룹 (2건) : ['공용/규정/복무규정_최종.md', '공용/규정/복무규정_최종_v2.md']
  그룹 (2건) : ['공용/매뉴얼/시스템매뉴얼.md', '임시/시스템매뉴얼_복사.md']


Step 2에서 파일 해시를 구해 두었는데도 **텍스트 해시**를 썼습니다.

이론 4.1절에서 본 대로 같은 내용이라도 저장한 도구가 다르면 파일 바이트가 다르고 반대로 형식이 달라도 추출한 텍스트는 같을 수 있습니다. **중복은 내용의 문제이므로 내용으로 판정합니다.**

In [38]:
# 이론 4.6절의 판단 기준을 적용해 대표 문서를 고릅니다
def pick_representative(group):
    """수정 시각, 경로, 파일명을 종합해 대표를 선택"""
    def score(c):
        s = 0
        if c["top"] != "임시":
            s += 10                            # 공식 폴더 우선
        if re.search(r"(final|최종|승인)", c["name"], re.I):
            s += 5
        m = re.search(r"v(\d+)", c["name"], re.I)
        if m:
            s += int(m.group(1))
        return (s, c["mtime"], c["size"])
    return max(group, key=score)


for h, group in exact_groups.items():
    rep = pick_representative(group)
    for c in group:
        if c is not rep:
            c["status"] = "duplicate"
            c["reject_reason"] = f"완전 중복. 대표: {rep['path']}"
            c["superseded_by"] = rep["path"]
            c["indexed"] = False
    print(f"대표 : {rep['path']}")

alive = [c for c in catalog if c["status"] == "parsed"]
print(f"\n남은 문서 {len(alive)}건")

대표 : 공용/규정/복무규정_최종_v2.md
대표 : 공용/매뉴얼/시스템매뉴얼.md

남은 문서 12건


대표를 고를 때 경로 문자열이 아니라 Step 2에서 카탈로그에 담아 둔 `c["top"]`으로 판단했습니다.

`c["path"].startswith("임시")`처럼 경로 문자열을 직접 자르면 **운영체제마다 구분 기호가 달라 조용히 틀립니다.** 판단에 쓸 값은 카탈로그를 만들 때 한 번 정해 두고 그것만 씁니다.

In [39]:
# 실습 : 근사 중복을 MinHash + LSH 2단계 구조로 찾으세요
#  - 1단계: LSH의 임계값을 여유 있게 낮춰(threshold - margin) 후보를 모읍니다
#  - 2단계: 후보 쌍만 실제 MinHash.jaccard() 유사도를 계산해 threshold 이상만 남깁니다

from datasketch import MinHash, MinHashLSH

NUM_PERM = 128          # 시그니처 길이. make_minhash 와 LSH 가 같은 값을 써야 합니다
SHINGLE_K = 3


def shingles(text, k=SHINGLE_K):
    t = re.sub(r"\s+", " ", text)
    return {t[i:i+k] for i in range(len(t) - k + 1)}


def make_minhash(text, num_perm=NUM_PERM):
    m = MinHash(num_perm=num_perm)
    for s in shingles(text):
        m.update(s.encode())
    return m


def find_near_duplicates(docs, threshold=0.7, margin=0.25):
    # 1단계 : LSH 로 후보만 모읍니다. 임계값을 낮춰 잡아 놓칠 위험을 줄입니다
    lsh = MinHashLSH(threshold=max(0.3, threshold - margin), num_perm=NUM_PERM)
    sigs = {}
    for c in docs:
        m = make_minhash(c["text"])
        sigs[c["path"]] = m
        lsh.insert(c["path"], m)

    # 2단계 : 후보만 실제 유사도로 다시 계산해 거릅니다
    seen, out = set(), []
    for c in docs:
        for other in lsh.query(sigs[c["path"]]):
            if other == c["path"]:
                continue
            key = tuple(sorted([c["path"], other]))
            if key in seen:
                continue
            seen.add(key)
            sim = sigs[key[0]].jaccard(sigs[key[1]])
            if sim >= threshold:
                out.append((key[0], key[1], sim))
    return sorted(out, key=lambda x: -x[2])

<details class="lms-answer">
<summary>풀이 보기</summary>

**빈칸 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 1단계 LSH 초기화
lsh = MinHashLSH(threshold=max(0.3, threshold - margin), num_perm=NUM_PERM)

# 2단계 정밀 자카드 유사도 계산
sim = sigs[key[0]].jaccard(sigs[key[1]])
```

<details>
<summary>전체 완성 코드 보기 (복사·붙여넣기용)</summary>

```python
def find_near_duplicates(docs, threshold=0.7, margin=0.25):
    # 1단계 : LSH 로 후보만 모읍니다. 임계값을 낮춰 잡아 놓칠 위험을 줄입니다
    lsh = MinHashLSH(threshold=max(0.3, threshold - margin), num_perm=NUM_PERM)
    sigs = {}
    for c in docs:
        m = make_minhash(c["text"])
        sigs[c["path"]] = m
        lsh.insert(c["path"], m)

    # 2단계 : 후보만 실제 유사도로 다시 계산해 거릅니다
    seen, out = set(), []
    for c in docs:
        for other in lsh.query(sigs[c["path"]]):
            if other == c["path"]:
                continue
            key = tuple(sorted([c["path"], other]))
            if key in seen:
                continue
            seen.add(key)
            sim = sigs[key[0]].jaccard(sigs[key[1]])
            if sim >= threshold:
                out.append((key[0], key[1], sim))
    return sorted(out, key=lambda x: -x[2])
```

</details>

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

이 코드는 **LSH를 최종 판정 함수가 아니라 1차 후보 생성기로 사용합니다.**

`MinHashLSH(threshold=0.7)`은 자카드 유사도 0.7을 경계로 정확히 자르는 장치가 아닙니다. 서명(Signature) 배열을 여러 밴드(구간)로 나누어 한 밴드라도 일치하면 후보로 올리는 확률적 해싱 방식이어서 **유사도가 0.7을 조금 넘는 쌍을 놓치거나(미탐) 0.7에 못 미치는 쌍을 후보로 올리기도(오탐) 합니다.** 따라서 LSH 단계에서는 임계값을 여유 있게 낮춰(`threshold - margin`) 후보를 넉넉히 모으고 최종 판정은 후보 쌍들의 실제 MinHash 자카드 유사도를 다시 계산해 정확히 필터링합니다.

이론 4.4절에서 설명한 **'저비용 해싱으로 후보를 좁히고 정밀 계산으로 확정하는 2단계 구조'**를 그대로 구현한 것입니다.

</details>

In [40]:
# 같은 데이터에 임계값만 바꿔 봅니다. 이론 4.5절이 요구한 확인입니다
for th in (0.7, 0.85):
    found = find_near_duplicates(alive, threshold=th) or []
    print(f"임계값 {th} : {len(found)}쌍")
    for a, b, sim in found:
        print(f"    {sim:.3f}  {a}  <->  {b}")

임계값 0.7 : 2쌍
    0.938  공용/규정/보안지침_1월.md  <->  공용/규정/보안지침_7월.md
    0.758  공용/규정/복무규정_2024.md  <->  공용/규정/복무규정_최종_v2.md
임계값 0.85 : 1쌍
    0.938  공용/규정/보안지침_1월.md  <->  공용/규정/보안지침_7월.md


유사도 임계값 `0.7`에서는 보안지침 쌍(1월·7월본)과 복무규정 쌍(2024년판·2026년판) 두 쌍이 모두 근사 중복으로 묶이지만 **임계값을 `0.85`로 높이면 복무규정 버전 쌍은 빠지고 보안지침 쌍 하나만 남습니다.**

이론 4.5절과 4.6절에서 짚은 것처럼 **복무규정 두 문서는 단순 복제물이 아니라 개정 연도가 다른 버전 관계**입니다. 임계값을 `0.7`로 낮춰 구버전을 자동 삭제해 버리면 "2024년 복무규정은 어땠나요?"라는 질문에 답할 수 없고, 반대로 `0.85`로 높이면 버전 문서는 보존되지만 일부 문단이 수정된 근사 중복 문서가 빠져나갈 수 있습니다. 결국 어느 쪽 오류를 감수할지 선택하는 문제이므로 **유사도 임계값 설정은 단순 파라미터 조정을 넘어 문서 보존 범위를 정하는 운영 정책 결정**입니다.

In [41]:
# 근사 중복은 지우지 않고 표시만 합니다. 이론 4.6절의 "삭제 대신 표시"
pairs = find_near_duplicates(alive, threshold=0.7) or []

for a, b, sim in pairs:
    doc_a = next(c for c in alive if c["path"] == a)
    doc_b = next(c for c in alive if c["path"] == b)
    older = doc_a if doc_a["mtime"] <= doc_b["mtime"] else doc_b
    newer = doc_b if older is doc_a else doc_a
    older["near_duplicate_of"] = newer["path"]
    older["near_duplicate_sim"] = round(sim, 3)
    older["indexed"] = False            # 검색에서 기본 제외. 삭제는 아닙니다
    print(f"{older['path']} 를 기본 제외로 표시 (유사도 {sim:.3f})")

final_docs = [c for c in alive if c["indexed"]]
print(f"\n최종 인덱싱 대상 {len(final_docs)}건 / 전체 {len(catalog)}건")

공용/규정/보안지침_1월.md 를 기본 제외로 표시 (유사도 0.938)
공용/규정/복무규정_2024.md 를 기본 제외로 표시 (유사도 0.758)

최종 인덱싱 대상 9건 / 전체 18건


### Step 7-1. 근사 중복 검출 기법(TF-IDF·MinHash·SimHash) 비교
교재 : 4.4.4 데이터 중복 제거하기

교재 4.4.4는 근사 중복을 찾는 방법으로 TF-IDF 유사도, MinHash, SimHash를 듭니다. MinHash 실습과 같은 문서 12건(`alive`)과 같은 3글자 shingle에 나머지 두 방법을 걸어 같은 쌍을 어떻게 판정하는지 나란히 봅니다. TF-IDF에 쓰는 scikit-learn은 코랩에 기본으로 설치되어 있고 사내 서버에서는 설치 목록에 넣습니다.

- **TF-IDF 코사인 유사도** : 문서마다 글자 3-그램에 TF-IDF 가중치를 매긴 벡터를 만들고 두 벡터의 코사인 유사도를 봅니다. 띄어쓰기가 흔들리는 한국어에는 단어보다 글자 단위가 맞습니다
- **SimHash** : shingle마다 64비트 해시를 만들어 비트가 1이면 +1, 0이면 -1을 더하고 합이 양수인 비트를 1로 두어 문서 하나를 64비트 값 하나로 줄입니다. 두 값에서 다른 비트의 수(해밍 거리)가 작을수록 비슷합니다

판정 기준은 MinHash와 TF-IDF에 교재의 0.8을 똑같이 쓰고 SimHash에는 64비트 SimHash에서 흔히 쓰는 해밍 거리 3 이하와 그보다 넉넉한 6 이하를 함께 봅니다.

In [42]:
# 교재 4.4.4의 세 방법을 MinHash 실습과 같은 문서, 같은 3글자 shingle 에 걸어 봅니다
from itertools import combinations
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

cmp_docs = list(alive)                    # 읽기만 합니다. 카탈로그는 바꾸지 않습니다

# 1) TF-IDF 코사인 유사도 : 글자 3-gram 벡터
tfidf = TfidfVectorizer(analyzer="char", ngram_range=(3, 3))
tfidf_cos = cosine_similarity(
    tfidf.fit_transform([re.sub(r"\s+", " ", c["text"]) for c in cmp_docs]))


# 2) SimHash : shingle 마다 64비트 해시를 만들고 비트별로 +1/-1 을 더해 합이 양수인 비트를 1로 둡니다
def simhash(text, bits=64):
    votes = [0] * bits
    for sh in shingles(text):
        h = int.from_bytes(hashlib.md5(sh.encode()).digest()[:8], "big")
        for k in range(bits):
            votes[k] += 1 if (h >> k) & 1 else -1
    return sum(1 << k for k in range(bits) if votes[k] > 0)


sim_fp = [simhash(c["text"]) for c in cmp_docs]
cmp_sigs = [make_minhash(c["text"]) for c in cmp_docs]
print(f"문서 {len(cmp_docs)}건, 비교할 쌍 {len(cmp_docs) * (len(cmp_docs) - 1) // 2}개")

문서 12건, 비교할 쌍 66개


In [43]:
# 쌍마다 세 점수를 구하고 교재 4.4.4의 0.8, SimHash 해밍 거리 3과 6으로 판정합니다
cmp_rows = []
for i, j in combinations(range(len(cmp_docs)), 2):
    cmp_rows.append((cmp_sigs[i].jaccard(cmp_sigs[j]), tfidf_cos[i, j],
                     bin(sim_fp[i] ^ sim_fp[j]).count("1"),
                     cmp_docs[i]["name"], cmp_docs[j]["name"]))
cmp_rows.sort(reverse=True)               # MinHash 유사도가 높은 쌍부터

print(f"{'MinHash':>7} {'TF-IDF':>7} {'해밍':>4}   M>=0.8 T>=0.8 S<=3  S<=6   문서 쌍")
for jac, tcos, ham, a, bb in cmp_rows[:3]:
    flags = [" 중복 " if hit else "  -   " for hit in (jac >= 0.8, tcos >= 0.8, ham <= 3, ham <= 6)]
    print(f"{jac:>7.3f} {tcos:>7.3f} {ham:>4}  {' '.join(flags)}  {a} <-> {bb}")

rest = cmp_rows[3:]
print(f"나머지 {len(rest)}쌍 : MinHash 최대 {max(r[0] for r in rest):.3f}, "
      f"TF-IDF 최대 {max(r[1] for r in rest):.3f}, 해밍 거리 최소 {min(r[2] for r in rest)}")

MinHash  TF-IDF   해밍   M>=0.8 T>=0.8 S<=3  S<=6   문서 쌍
  0.938   0.938    5   중복   중복    -     중복   보안지침_1월.md <-> 보안지침_7월.md
  0.758   0.891   10    -     중복    -      -     복무규정_2024.md <-> 복무규정_최종_v2.md
  0.156   0.187   25    -      -      -      -     비상연락망.md <-> 2026상반기_인사평가.md
나머지 63쌍 : MinHash 최대 0.141, TF-IDF 최대 0.225, 해밍 거리 최소 20


**같은 0.8이라도 방법마다 엄격함이 다릅니다.** MinHash(자카드 유사도의 추정)는 0.8에서 보안지침 쌍만 묶고 코사인 유사도는 같은 쌍에서 대개 자카드 유사도보다 높게 나오는 척도라서 TF-IDF는 복무규정 버전 쌍까지 묶습니다. SimHash의 해밍 거리는 두 문서가 다른 정도에 비례해 커집니다. 0.94쯤 겹치는 보안지침 쌍도 64비트 가운데 5비트가 다르므로 해밍 거리 3은 거의 같은 문서만 잡는 기준입니다. 그래서 임계값은 방법마다 따로 정하고 이미 아는 중복 쌍과 버전 쌍으로 확인합니다.

비용은 문서마다 64비트 값 하나만 저장하고 비교하는 SimHash가 세 방법 중 가장 연산 비용이 낮고 TF-IDF 점수는 말뭉치 전체의 역문서 빈도(IDF, 그 글자 조각이 몇 개 문서에 나오는지)에 달려 있어 문서가 늘면 다시 계산해야 합니다.

## Step 8. 전처리 통계 보고서 및 정제 스냅샷 저장

이론 6.2절에서 단계별 입력과 출력 건수가 모든 지표의 뼈대라고 했습니다. 그 건수를 출력합니다.

In [44]:
print(f"{'상태':<16} {'건수':>6}")
print("-" * 24)
for status, n in Counter(c["status"] for c in catalog).most_common():
    print(f"{status:<16} {n:>6}")

print(f"\n{'제외와 보류 사유':<50}")
print("-" * 76)
for c in catalog:
    if c["reject_reason"]:
        tail = f"  [재시도 {c['retry_path']}]" if c["retry_path"] else ""
        print(f"  {c['path']:<40} {c['reject_reason']}{tail}")

print(f"\n{'형식별 건수와 추출 성공률':<40}")
print("-" * 44)
by_ext = {}
for c in catalog:
    ok, tot = by_ext.get(c["ext"], (0, 0))
    by_ext[c["ext"]] = (ok + (1 if c["char_len"] > 0 else 0), tot + 1)
for ext, (ok, tot) in sorted(by_ext.items()):
    print(f"  {ext:<8} {ok}/{tot}  ({ok/tot:.0%})")

상태                   건수
------------------------
parsed               12
filtered              3
duplicate             2
needs_ocr             1

제외와 보류 사유                                         
----------------------------------------------------------------------------
  공용/규정/복무규정_최종.md                         완전 중복. 대표: 공용/규정/복무규정_최종_v2.md
  영업/계약/시설이용계약서_스캔본.pdf                    글자 0자, 파일 30,003B. 스캔본 의심  [재시도 OCR]
  인사/평가/2026상반기_인사평가.md                    기밀 등급 문서. 권한 분리 후 재검토
  임시/구결재기록_2019.txt                        대체 문자 비율 19%
  임시/시스템매뉴얼_복사.md                          완전 중복. 대표: 공용/매뉴얼/시스템매뉴얼.md
  임시/신규문서.md                               본문 부족 (0자)
  임시/템플릿.md                                상투적 문구(보일러플레이트) 비율 45%

형식별 건수와 추출 성공률                          
--------------------------------------------
  .docx    1/1  (100%)
  .hwpx    1/1  (100%)
  .md      11/12  (92%)
  .pdf     1/2  (50%)
  .txt     1/1  (100%)
  .xlsx    1/1  (100%)


형식별 추출 성공률에서 `.pdf`가 2건 중 1건이라 이론 6.3절 표의 "**특정 형식만 추출 성공률이 낮음**"이 잡힙니다.

우리는 원인이 스캔본 한 건임을 이미 알지만 실무에서는 원인을 모르는 채 이 수치를 먼저 보므로 **표본을 열어 원인을 확인하는 일**이 다음 순서입니다.

In [45]:
# 카탈로그와 정제 문서를 파일로 저장합니다
def serializable(c):
    return {k: v for k, v in c.items() if k != "text"}


with open("catalog.json", "w", encoding="utf-8") as f:
    json.dump([serializable(c) for c in catalog], f, ensure_ascii=False, indent=2)

CLEAN = Path("corpus_clean")
if CLEAN.exists():
    shutil.rmtree(CLEAN)
CLEAN.mkdir()

for c in final_docs:
    text = mask_pii(c["text"]) if c.get("policy") == "mask" else c["text"]
    out = CLEAN / (c["path"].replace("/", "__") + ".txt")
    out.write_text(text, encoding="utf-8")

print("catalog.json 과 corpus_clean/ 저장 완료")
for p in sorted(CLEAN.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size:,} B)")

catalog.json 과 corpus_clean/ 저장 완료
  공용__규정__공문처리지침.hwpx.txt  (451 B)
  공용__규정__교육비지원규정.pdf.txt  (686 B)
  공용__규정__보안지침_7월.md.txt  (650 B)
  공용__규정__복무규정_최종_v2.md.txt  (781 B)
  공용__규정__출장여비규정.docx.txt  (522 B)
  공용__매뉴얼__비상연락망.md.txt  (423 B)
  공용__매뉴얼__시스템매뉴얼.md.txt  (466 B)
  공용__부서별예산.xlsx.txt  (166 B)
  영업__계약__협력사계약현황.md.txt  (315 B)


## Step 9. 평가·학습용 합성 데이터 생성

교재 : 4.3.1 합성 데이터, 4.5.3 파인 튜닝 데이터셋 생성, 4.5.4 지시 파인 튜닝 데이터셋 자동 생성

정제된 문서로 이론 6.4절의 세 번째 방법인 **모델로 생성하기**로 합성 데이터를 만듭니다. 용도는 RAG 평가 세트와 파인튜닝 데이터셋 두 가지입니다.

교재 4.5.4는 문서에서 질문을 만들고 검색으로 찾은 청크에서 답을 고르는 방식을 권하고 자원이 부족하면 검색을 생략하고 한 문단에서 질문과 답을 모두 만드는 축소판을 제시합니다. 이 Step은 검색을 생략한 그 축소판에 가깝고(문단 대신 짧은 문서 한 건) 검색을 쓰는 방식은 데이터셋 자동화 노트에서 다룹니다.

**여기서부터 GPU가 필요합니다.**

In [46]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-3B-Instruct"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb,
    dtype=torch.float16,          # torch_dtype= 는 폐기되었습니다
    device_map="auto",
)
model.eval()
print(f"생성 모델 로드 완료. VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

생성 모델 로드 완료. VRAM 2.06 GB


`dtype=`을 썼습니다(예전의 `torch_dtype=`은 폐기됨). `bnb_4bit_compute_dtype`은 가중치는 4비트로 담고 **계산은 16비트로** 한다는 다른 설정입니다.

In [47]:
SYSTEM_PROMPT = (
    "당신은 사내 문서를 다루는 데이터 담당자입니다. "
    "요청한 형식만 출력하고, 답변은 반드시 한국어로 작성합니다."
)


def gen(prompt, max_new_tokens=900, temperature=0.7):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    ids = tok(text, return_tensors="pt").to(model.device)

    # 샘플링일 때만 temperature 를 넘깁니다. 안 그러면 호출마다 경고가 납니다
    kw = ({"do_sample": True, "temperature": temperature, "top_p": 0.9}
          if temperature > 0 else {"do_sample": False})

    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens,
                             pad_token_id=tok.eos_token_id, **kw)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

시스템 프롬프트에 "**답변은 반드시 한국어로**"를 넣었습니다. 한국어 문서를 주면 한국어로 답할 것 같지만 여러 언어로 학습한 모델은 프롬프트가 길어지면 영어로 새기도 하고 한 응답 안에서 언어가 섞이기도 합니다. **출력 언어는 지정해야 고정됩니다.**

생성 프롬프트에는 이론 6.4절이 반드시 넣으라고 한 세 장치를 그대로 넣습니다.

- 질문의 종류를 강제로 나눕니다. 별도 제약을 두지 않으면 모델은 쉬운 질문만 만듭니다  
- 근거를 원문에서 그대로 인용하게 합니다. 나중에 기계로 확인할 수 있습니다  
- 원문 표현을 그대로 베끼지 못하게 합니다. 문서와 똑같은 말로 물으면 검색이 너무 쉬워집니다

In [48]:
QA_PROMPT = """다음 사내 문서를 읽고 질문과 답변 쌍 3개를 만드세요.

규칙
1. 세 질문은 서로 다른 유형이어야 합니다.
   - 1번: 단순 조회 (문서에 명시된 사실을 묻기)
   - 2번: 조건 판단 (예외나 조건이 있는 규정을 묻기)
   - 3번: 종합 (여러 항목을 함께 봐야 답할 수 있는 것)
2. 질문은 문서의 표현을 그대로 쓰지 말고 일상적인 말투로 쓰세요.
3. 답변은 문서 내용에만 근거해야 합니다. 추측을 넣지 마세요.
4. 각 쌍에 근거가 되는 원문 문장을 그대로 인용하세요.
5. 질문과 답변과 근거를 모두 한국어로 쓰세요.
6. JSON 배열로만 출력하세요. 설명을 붙이지 마세요.

형식
[{{"type": "단순조회", "question": "", "answer": "", "evidence": ""}}]

[문서]
{doc}"""

# 문서를 자를 길이를 글자 수로 하드코딩하지 않고 토크나이저에서 읽어 정합니다
DOC_TOKEN_BUDGET = 1200


def fit_doc(text, budget=DOC_TOKEN_BUDGET):
    ids = tok(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= budget:
        return text
    return tok.decode(ids[:budget])


def parse_json_array(raw):
    """(항목리스트, 실패사유) 를 반환합니다. 예외를 던지지 않습니다"""
    cleaned = re.sub(r"```(json)?", "", raw).strip()
    m = re.search(r"\[.*\]", cleaned, re.S)
    if not m:
        return None, "JSON 배열을 찾지 못함"
    try:
        items = json.loads(m.group(0))
    except json.JSONDecodeError as e:
        return None, f"JSON 파싱 실패: {e.msg}"
    if not isinstance(items, list):
        return None, "배열이 아님"
    return items, None

문서를 프롬프트에 넣을 때 `text[:2500]`처럼 글자 수로 고정해 자르지 않고 **모델의 토크나이저로 실제 토큰 수를 계산해 컨텍스트 한도에 맞춰 잘랐습니다.** 한국어는 모델마다 글자당 토큰화 비율이 크게 달라서 글자 수로 고정하면 모델을 교체할 때 컨텍스트가 남거나 초과되기 때문입니다. 또한 `parse_json_array` 함수가 예외를 발생시키는 대신 `(결과, 사유)` 튜플을 반환하도록 작성하여 **생성·파싱 실패를 예외 중단이 아닌 정상 파이프라인의 분기로 처리**했습니다.

In [49]:
MIN_GEN_CHARS = 100      # 이보다 짧은 문서에서는 질문 3개가 안 나옵니다

synth, gen_fail, gen_skip = [], [], []

for c in final_docs:
    text = mask_pii(c["text"]) if c.get("policy") == "mask" else c["text"]
    if len(text) < MIN_GEN_CHARS:
        gen_skip.append((c["path"], f"본문 {len(text)}자"))
        continue
    raw = gen(QA_PROMPT.format(doc=fit_doc(text)))
    items, err = parse_json_array(raw)
    if items is None:
        gen_fail.append((c["path"], err))
        print(f"[생성 실패] {c['path']} : {err}")
        continue
    for it in items:
        if not isinstance(it, dict):
            continue
        it["source"] = c["path"]
        it["source_text"] = text
        synth.append(it)

print(f"\n생성 {len(synth)}쌍, 실패 {len(gen_fail)}건, 건너뜀 {len(gen_skip)}건")
for path, why in gen_fail + gen_skip:
    print(f"  {path} : {why}")
for s in synth[:3]:
    print(f"\n[{s.get('type')}] {s.get('question')}")
    print(f"  답변 : {s.get('answer')}")
    print(f"  근거 : {str(s.get('evidence', ''))[:80]}")

[생성 실패] 공용/규정/교육비지원규정.pdf : JSON 파싱 실패: Extra data
[생성 실패] 공용/매뉴얼/비상연락망.md : JSON 파싱 실패: Extra data

생성 19쌍, 실패 2건, 건너뜀 0건
  공용/규정/교육비지원규정.pdf : JSON 파싱 실패: Extra data
  공용/매뉴얼/비상연락망.md : JSON 파싱 실패: Extra data

[단순조회] 공문 접수는 어디에서 이루어집니까?
  답변 : 총무팀에서 공문을 접수합니다.
  근거 : 제1조 (접수) 외부 기관에서 온 공문은 총무팀이 일괄 접수한다.

[단순조회] 공문은 얼마나 오래 보존합니까?
  답변 : 5년간 보존합니다.
  근거 : 제5조 (보존) 접수한 공문의 원본은 5년간 보존한다.

[조건 판단] 대기 중인 공문은 회신을 받을까요?
  답변 : 회신이 필요한 공문에 한해 회신한다.
  근거 : 제4조 (회신) 회신이 필요한 공문은 접수일로부터 14일 이내에 회신한다.


3B 규모의 소형 모델은 출력 시 JSON 문법을 종종 깨뜨리므로 파싱 실패가 몇 건 발생할 수 있습니다. 이때 **JSON 파싱 실패(`fails`)와 본문 길이 부족으로 인한 건너뜀(`skipped`)은 후속 조치가 다르므로 반드시 분리해 경로·사유와 함께 기록합니다.** JSON 형식이 깨진 문서는 재생성을 시도하면 해결되지만 본문이 짧아 건너뛴 문서는 몇 번을 다시 실행해도 동일하므로 건너뛴 목록을 기록해 두지 않으면 나중에 "특정 규정의 평가 문항이 왜 하나도 없는가"라는 문의에 답할 수 없습니다.

## Step 10. 합성 데이터 정합성 검증

교재 : 4.5.4 지시 파인 튜닝 데이터셋 자동 생성

이론 6.5절의 **네 가지 최소 검증**을 센서로 구현합니다. 전부 코드로 셀 수 있습니다.

- 형식이 맞는가. 필요한 항목이 다 있는가  
- 근거가 원문에 있는가  
- 원문에 없는 숫자가 답에 있는가  
- 질문이 원문 표현을 그대로 베꼈는가

교재 4.5.4가 생성 뒤에 권하는 세 가지 검사(거의 같은 질문-답변 쌍 제거, 검색된 문단에 없는 정보를 넣은 답변 제거, 무작위 소규모 표본 직접 점검) 가운데 앞의 둘이 아래의 근사 중복 검사와 근거·숫자 검사입니다. 교재는 근사 중복에 임베딩 코사인 유사도를 쓰고 여기서는 Step 7의 MinHash를 다시 씁니다.

In [55]:
# 실습 : 합성 질의응답(QA) 쌍의 품질을 검증하는 함수를 완성하세요
#  1) 필수 필드 누락 검사 (미리 작성됨)
#  2) 근거(evidence)가 원문(src)에 실제로 존재하는지 검사 (빈칸 1)
#  3) 답변(a)에 원문(src)에 없는 숫자가 들어갔는지(환각) 검사 (빈칸 2)
#  4) 질문(q)이 원문 표현을 COPY_RUN 글자 이상 그대로 베꼈는지 검사 (미리 작성됨)

REQUIRED_FIELDS = ("type", "question", "answer", "evidence")
COPY_RUN = 10          # 질문이 원문과 몇 글자 이상 연속으로 겹치면 베낀 것으로 볼지


def normalize(s):
    return re.sub(r"\s+", "", s or "")


def validate_pair(item):
    issues = []
    src = item.get("source_text", "")

    # 1) 형식이 맞는가
    for f in REQUIRED_FIELDS:
        if not item.get(f):
            issues.append(f"필수필드누락: {f}")
    if issues:
        return {"pass": False, "issues": issues}

    q, a, ev = item["question"], item["answer"], item["evidence"]

    # 2) 근거가 원문에 있는가
    if normalize(ev) not in normalize(src):
        issues.append("근거불일치: evidence 가 원문에 없음")

    # 3) 원문에 없는 숫자가 답에 있는가
    hallucinated = set(re.findall(r"\d+", a)) - set(re.findall(r"\d+", src))
    if hallucinated:
        issues.append(f"환각의심: 원문에 없는 숫자 {sorted(hallucinated)}")

    # 4) 질문이 원문 표현을 그대로 베꼈는가
    nq, ns = normalize(q), normalize(src)
    for i in range(max(len(nq) - COPY_RUN, 0)):
        if nq[i:i+COPY_RUN] in ns:
            issues.append(f"표현복사: 질문이 원문 표현을 {COPY_RUN}자 이상 그대로 사용")
            break

    return {"pass": len(issues) == 0, "issues": issues}

<details class="lms-answer">
<summary>풀이 보기</summary>

**빈칸 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 2) 근거가 원문에 있는가
if normalize(ev) not in normalize(src):
    issues.append("근거불일치: evidence 가 원문에 없음")

# 3) 원문에 없는 숫자가 답에 있는가
ans_nums = set(re.findall(r"\d+", a))
src_nums = set(re.findall(r"\d+", src))
hallucinated = ans_nums - src_nums
```

<details>
<summary>전체 완성 코드 보기 (복사·붙여넣기용)</summary>

```python
def validate_pair(item):
    issues = []
    src = item.get("source_text", "")

    # 1) 형식이 맞는가
    for f in REQUIRED_FIELDS:
        if not item.get(f):
            issues.append(f"필수필드누락: {f}")
    if issues:
        return {"pass": False, "issues": issues}

    q, a, ev = item["question"], item["answer"], item["evidence"]

    # 2) 근거가 원문에 있는가
    if normalize(ev) not in normalize(src):
        issues.append("근거불일치: evidence 가 원문에 없음")

    # 3) 원문에 없는 숫자가 답에 있는가
    hallucinated = set(re.findall(r"\d+", a)) - set(re.findall(r"\d+", src))
    if hallucinated:
        issues.append(f"환각의심: 원문에 없는 숫자 {sorted(hallucinated)}")

    # 4) 질문이 원문 표현을 그대로 베꼈는가
    nq, ns = normalize(q), normalize(src)
    for i in range(max(len(nq) - COPY_RUN, 0)):
        if nq[i:i+COPY_RUN] in ns:
            issues.append(f"표현복사: 질문이 원문 표현을 {COPY_RUN}자 이상 그대로 사용")
            break

    return {"pass": len(issues) == 0, "issues": issues}
```

</details>

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

**판정 함수의 기본값은 불합격입니다.** 그래서 빈 함수를 실행했을 때 통과율이 100퍼센트가 아니라 0퍼센트로 나왔습니다. 덜 만든 판정 함수가 모든 데이터를 합격시키면 아무도 눈치채지 못한 채 불량 데이터가 흘러갑니다.

숫자 검사가 특히 쓸모 있습니다. **원문에 없는 숫자가 답변에 있으면 환각이고** 규정 문서에서는 일수와 금액과 시각이 틀리면 답 전체가 틀린 것이 됩니다.

다만 이론 6.5절에서 본 대로 **이 검사들에도 오탐이 있습니다.** 원문의 "2026년"을 보고 계산해서 "2027년"이라고 쓴 답은 환각으로 잡힙니다. 계산이 필요한 데이터에서는 이 검사를 느슨하게 하고 **검사도 틀린다는 것을 알고 씁니다.**

</details>

In [51]:
results = [validate_pair(s) for s in synth]
n_pass = sum(r["pass"] for r in results)
print(f"통과 {n_pass}/{len(synth)} ({n_pass/max(len(synth),1):.0%})")

issue_counter = Counter()
for r in results:
    for i in r["issues"]:
        issue_counter[i.split(":")[0]] += 1
print("문제 유형 :", dict(issue_counter))

통과 14/19 (74%)
문제 유형 : {'표현복사': 2, '근거불일치': 3}


In [52]:
clean_synth = [s for s, r in zip(synth, results) if r["pass"]]

# 다양성 확인. 이론 6.5절의 세 번째 위험(표현 단조로움)에 대한 대응입니다
types = Counter(s.get("type") for s in clean_synth)
q_lens = [len(s["question"]) for s in clean_synth] or [0]
uniq_words = len({w for s in clean_synth for w in re.findall(r"[가-힣]+", s["question"])})

print(f"통과 데이터 {len(clean_synth)}쌍")
print(f"유형 분포 : {dict(types)}")
print(f"질문 길이 : 평균 {np.mean(q_lens):.0f}자, 범위 {min(q_lens)}에서 {max(q_lens)}")
print(f"질문 고유 어휘 수 : {uniq_words}")

통과 데이터 14쌍
유형 분포 : {'단순조회': 11, '조건 판단': 1, '종합': 2}
질문 길이 : 평균 21자, 범위 15에서 29
질문 고유 어휘 수 : 60


In [53]:
# 근사 중복 검사를 합성 데이터에도 적용합니다. 같은 질문이 반복 생성되는 일이 흔합니다
q_sigs, dups = {}, 0
for s in clean_synth:
    m = make_minhash(s["question"])
    for other_q, other_m in q_sigs.items():
        if m.jaccard(other_m) > 0.8:
            dups += 1
            s["near_dup_of"] = other_q
            break
    else:
        q_sigs[s["question"]] = m

final_synth = [s for s in clean_synth if "near_dup_of" not in s]
print(f"근사 중복 질문 {dups}쌍")
print(f"최종 데이터 {len(final_synth)}쌍")

근사 중복 질문 0쌍
최종 데이터 14쌍


In [54]:
with open("synthetic_qa.json", "w", encoding="utf-8") as f:
    json.dump([{k: v for k, v in s.items() if k != "source_text"} for s in final_synth],
              f, ensure_ascii=False, indent=2)
print("synthetic_qa.json 저장 완료")

synthetic_qa.json 저장 완료


생성 건수보다 최종 통과 건수가 상당히 줄었을 텐데 **이것이 정상입니다.** 합성 데이터는 많이 만들고 많이 버리는 것이 전제라 통과율 50퍼센트면 필요량의 두 배를 생성해야 합니다.

통과율 자체도 지표입니다. 통과율이 낮으면 생성 프롬프트를 고쳐야 하고 **통과율이 100퍼센트에 가까우면 검증이 너무 느슨한 것입니다.** 이론 6.3절 표의 "모든 단계 통과율이 아주 높음"이 이 경우입니다.

## Step 11. 선택 심화 트랙

트랙 2\~4 중 **두 개를 선택**해 수행하고 공유합니다.

<div style="font-size: 18px; line-height: 1.6;">

| 트랙 | 주제 | 성격 | 난이도 |
|---|---|---|---|
| 1 | 임베딩 기반 근사 중복 탐지 | 측정 | 중 |
| 2 | 생성 프롬프트 개선으로 통과율 올리기 | 개선 | 중 |
| 3 | 버전 판별과 유효 문서 결정 | 개선 | 상 |

</div>

#### 트랙 1. 증분 수집과 삭제 조정
*(안내: 증분 수집과 삭제 동기화는 파이프라인 노트(D42) 본 실습 Step 10으로 통합되어 진행합니다.)*

#### 트랙 2. 임베딩 코사인 유사도 기반 의미 중복 검출

MinHash는 **표현이 비슷한 것**을, 임베딩은 **뜻이 비슷한 것**을 잡습니다. 이론 4.4절에서 두 방법이 서로 다른 것을 잡는다고 한 것을 직접 확인합니다.

In [56]:
from langchain_huggingface import HuggingFaceEmbeddings

# 뜻은 같지만 표현이 다른 문서를 추가합니다
paraphrase = """# 재택근무 안내

직원은 일주일에 두 번까지 집에서 일할 수 있습니다.
집에서 일하려면 하루 전 저녁 여섯시까지 팀장에게 허락을 받아야 합니다.
"""
w("공용/매뉴얼/재택근무안내.md", paraphrase)

device = "cuda" if torch.cuda.is_available() else "cpu"
emb = HuggingFaceEmbeddings(model_name="BAAI/bge-m3",
                            model_kwargs={"device": device},
                            encode_kwargs={"normalize_embeddings": True})

docs = [{"path": c["path"], "text": c["text"]} for c in final_docs]
docs.append({"path": "공용/매뉴얼/재택근무안내.md", "text": paraphrase})

vecs = np.array(emb.embed_documents([d["text"][:2000] for d in docs]))
sim = vecs @ vecs.T

# 임계값을 정하지 않고 임베딩 유사도 상위 여덟 쌍만 봅니다.
# 임베딩 모델마다 유사도의 바닥값이 달라서 임계값을 고정값으로 하드코딩하면 모델을 바꿀 때 맞지 않습니다
rows = []
for i in range(len(docs)):
    for j in range(i + 1, len(docs)):
        js = make_minhash(docs[i]["text"]).jaccard(make_minhash(docs[j]["text"]))
        rows.append((float(sim[i, j]), js, docs[i]["path"], docs[j]["path"]))

print(f"{'임베딩':>7} {'MinHash':>8}   문서 쌍")
for e, js, a, b in sorted(rows, reverse=True)[:8]:
    print(f"{e:>7.3f} {js:>8.3f}   {a} <-> {b}")

/usr/local/lib/python3.13/dist-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.7.0)/charset_normalizer (3.4.9) doesn't match a supported version!
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

    임베딩  MinHash   문서 쌍
  0.697    0.055   공용/규정/복무규정_최종_v2.md <-> 공용/매뉴얼/재택근무안내.md
  0.688    0.086   공용/규정/보안지침_7월.md <-> 공용/매뉴얼/시스템매뉴얼.md
  0.672    0.117   공용/규정/교육비지원규정.pdf <-> 공용/규정/복무규정_최종_v2.md
  0.645    0.062   공용/규정/복무규정_최종_v2.md <-> 공용/규정/출장여비규정.docx
  0.622    0.102   공용/규정/교육비지원규정.pdf <-> 공용/규정/출장여비규정.docx
  0.612    0.023   공용/규정/공문처리지침.hwpx <-> 공용/규정/보안지침_7월.md
  0.584    0.000   공용/규정/공문처리지침.hwpx <-> 공용/매뉴얼/시스템매뉴얼.md
  0.562    0.008   공용/규정/출장여비규정.docx <-> 공용/매뉴얼/재택근무안내.md


<details class="lms-answer">
<summary>풀이 보기</summary>

**빈칸 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 1) 코사인 유사도 행렬 계산
sim = vecs @ vecs.T

# 2) MinHash 자카드 유사도 계산
js = make_minhash(docs[i]["text"]).jaccard(make_minhash(docs[j]["text"]))
```

<details>
<summary>전체 완성 코드 보기 (복사·붙여넣기용)</summary>

```python
from langchain_huggingface import HuggingFaceEmbeddings

# 뜻은 같지만 표현이 다른 문서를 추가합니다
paraphrase = """# 재택근무 안내

직원은 일주일에 두 번까지 집에서 일할 수 있습니다.
집에서 일하려면 하루 전 저녁 여섯시까지 팀장에게 허락을 받아야 합니다.
"""
w("공용/매뉴얼/재택근무안내.md", paraphrase)

device = "cuda" if torch.cuda.is_available() else "cpu"
emb = HuggingFaceEmbeddings(model_name="BAAI/bge-m3",
                            model_kwargs={"device": device},
                            encode_kwargs={"normalize_embeddings": True})

docs = [{"path": c["path"], "text": c["text"]} for c in final_docs]
docs.append({"path": "공용/매뉴얼/재택근무안내.md", "text": paraphrase})

vecs = np.array(emb.embed_documents([d["text"][:2000] for d in docs]))
sim = vecs @ vecs.T

# 임계값을 정하지 않고 임베딩 유사도 상위 여덟 쌍만 봅니다.
# 임베딩 모델마다 유사도의 바닥값이 달라서 임계값을 고정값으로 하드코딩하면 모델을 바꿀 때 맞지 않습니다
rows = []
for i in range(len(docs)):
    for j in range(i + 1, len(docs)):
        js = make_minhash(docs[i]["text"]).jaccard(make_minhash(docs[j]["text"]))
        rows.append((float(sim[i, j]), js, docs[i]["path"], docs[j]["path"]))

print(f"{'임베딩':>7} {'MinHash':>8}   문서 쌍")
for e, js, a, b in sorted(rows, reverse=True)[:8]:
    print(f"{e:>7.3f} {js:>8.3f}   {a} <-> {b}")
```

</details>

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

**재택근무안내와 복무규정은 같은 내용을 다른 어휘로 써서 임베딩 유사도는 높은데 MinHash 유사도는 바닥입니다.**

이 둘은 중복일까요? 내용은 같지만 하나는 규정 원문이고 하나는 쉬운 설명이라 **용도가 다르므로 둘 다 남기는 것이 나을 수 있습니다.** 정확한 조문을 원하는 사람에게는 복무규정이, 쉬운 안내를 원하는 사람에게는 재택근무안내가 맞는 답입니다.

실무에서는 임베딩 중복 판정을 **자동 삭제가 아니라 검토 대상 표시**로만 씁니다.

</details>

#### 트랙 3. 생성 프롬프트 개선과 검증 통과율 비교

컨텍스트와 하네스 노트에서 배운 방식대로 프롬프트를 고쳐 Step 10의 통과율을 끌어올립니다.

In [57]:
fail_types = Counter()
for r in results:
    for i in r["issues"]:
        fail_types[i.split(":")[0]] += 1
print("실패 유형 :", dict(fail_types))

QA_PROMPT_V2 = QA_PROMPT.replace(
    "6. JSON 배열로만 출력하세요. 설명을 붙이지 마세요.",
    """6. evidence 는 원문에서 문장을 글자 하나도 바꾸지 않고 복사해 넣으세요.
7. 답변에 숫자를 쓸 때는 원문에 있는 숫자만 쓰세요. 계산하거나 추정하지 마세요.
8. 질문에는 원문의 표현을 열 글자 이상 연속으로 쓰지 마세요. 뜻이 같은 다른 말로 바꾸세요.
9. JSON 배열로만 출력하세요. 설명이나 코드블록 표시를 붙이지 마세요."""
)

def run_generation(prompt_tpl, docs, temperature=0.7):
    out, fails = [], []
    for c in docs:
        text = mask_pii(c["text"]) if c.get("policy") == "mask" else c["text"]
        if len(text) < MIN_GEN_CHARS:
            continue
        items, err = parse_json_array(
            gen(prompt_tpl.format(doc=fit_doc(text)), temperature=temperature))
        if items is None:
            fails.append((c["path"], err))
            continue
        for it in items:
            if isinstance(it, dict):
                it["source"] = c["path"]
                it["source_text"] = text
                out.append(it)
    return out, fails

synth_v2, fails_v2 = run_generation(QA_PROMPT_V2, final_docs)
res_v2 = [validate_pair(s) for s in synth_v2]
p1 = sum(r["pass"] for r in results) / max(len(results), 1)
p2 = sum(r["pass"] for r in res_v2) / max(len(res_v2), 1)
print(f"\n기존 프롬프트 : 생성 {len(synth)}쌍, 통과율 {p1:.0%}")
print(f"개선 프롬프트 : 생성 {len(synth_v2)}쌍, 통과율 {p2:.0%}")

실패 유형 : {'표현복사': 2, '근거불일치': 3}

기존 프롬프트 : 생성 19쌍, 통과율 74%
개선 프롬프트 : 생성 15쌍, 통과율 67%


In [58]:
fail_types = Counter()
for r in results:
    for i in r["issues"]:
        fail_types[i.split(":")[0]] += 1
print("실패 유형 :", dict(fail_types))

QA_PROMPT_V2 = QA_PROMPT.replace(
    "6. JSON 배열로만 출력하세요. 설명을 붙이지 마세요.",
    """6. evidence는 원문에 있는 문장을 그대로 복사해 넣으세요.
7. 질문은 원문 문장을 베끼지 말고, 일반 사용자가 묻는 것처럼 다른 말로 바꾸어 쓰세요.
8. JSON 배열로만 출력하세요. 설명이나 코드블록 표시를 붙이지 마세요.""",
)

def run_generation(prompt_tpl, docs, temperature=0.7):
    out, fails = [], []
    for c in docs:
        text = mask_pii(c["text"]) if c.get("policy") == "mask" else c["text"]
        if len(text) < MIN_GEN_CHARS:
            continue
        items, err = parse_json_array(
            gen(prompt_tpl.format(doc=fit_doc(text)), temperature=temperature))
        if items is None:
            fails.append((c["path"], err))
            continue
        for it in items:
            if isinstance(it, dict):
                it["source"] = c["path"]
                it["source_text"] = text
                out.append(it)
    return out, fails

synth_v2, fails_v2 = run_generation(
    QA_PROMPT_V2, final_docs, temperature=0.2
)
res_v2 = [validate_pair(s) for s in synth_v2]

p1 = sum(r["pass"] for r in results) / max(len(results), 1)
p2 = sum(r["pass"] for r in res_v2) / max(len(res_v2), 1)
print(f"\n기존 프롬프트 : 생성 {len(synth)}쌍, 통과율 {p1:.0%}")
print(f"개선 프롬프트 : 생성 {len(synth_v2)}쌍, 통과율 {p2:.0%}")

실패 유형 : {'표현복사': 2, '근거불일치': 3}

기존 프롬프트 : 생성 19쌍, 통과율 74%
개선 프롬프트 : 생성 28쌍, 통과율 57%


In [59]:
from collections import Counter

# 개선 프롬프트(res_v2)의 실패 유형 집계
fail_types_v2 = Counter()
for r in res_v2:
    for i in r["issues"]:
        fail_types_v2[i.split(":")[0]] += 1

print("개선 프롬프트 실패 유형 :", dict(fail_types_v2))

# 실제로 탈락한 데이터 2개만 눈으로 확인해보기
print("\n--- [탈락한 데이터 샘플] ---")
count = 0
for s, r in zip(synth_v2, res_v2):
    if not r["pass"]:
        print(f"탈락 사유 : {r['issues']}")
        print(f"질문 : {s.get('question')}")
        print(f"답변 : {s.get('answer')}")
        print(f"근거 : {s.get('evidence')}")
        print("-" * 40)
        count += 1
        if count >= 2:
            break

개선 프롬프트 실패 유형 : {'근거불일치': 6, '필수필드누락': 4, '표현복사': 2}

--- [탈락한 데이터 샘플] ---
탈락 사유 : ['근거불일치: evidence 가 원문에 없음']
질문 : 연간 지원 한도는 얼마인가요?
답변 : 연간 200만원을 한도로 한다.
근거 : 제3조 (지원 한도) 1인당 연간 200만원을 한도로 한다.
----------------------------------------
탈락 사유 : ['근거불일치: evidence 가 원문에 없음']
질문 : 어학 과정은 연간 얼마나 지원받을 수 있나요?
답변 : 연간 100만원을 한도로 한다.
근거 : 제3조 (지원 한도) 어학 과정은 연간 100만원을 한도로 한다.
----------------------------------------


In [63]:
fail_types = Counter()
for r in results:
    for i in r["issues"]:
        fail_types[i.split(":")[0]] += 1
print("실패 유형 :", dict(fail_types))

QA_PROMPT_V2 = QA_PROMPT.replace(
    "6. JSON 배열로만 출력하세요. 설명을 붙이지 마세요.",
    """6. evidence는 본문에 존재하는 한 문장을 있는 그대로 복사하세요.
7. question은 일반 사용자의 질문 말투로 바꾸세요.
8. 설명, 인사말, 마크다운 코드블록 없이 오직 순수한 JSON 배열만 출력하세요.""",
)

def run_generation(prompt_tpl, docs, temperature=0.7):
    out, fails = [], []
    for c in docs:
        text = mask_pii(c["text"]) if c.get("policy") == "mask" else c["text"]
        if len(text) < MIN_GEN_CHARS:
            continue
        items, err = parse_json_array(
            gen(prompt_tpl.format(doc=fit_doc(text)), temperature=temperature))
        if items is None:
            fails.append((c["path"], err))
            continue
        for it in items:
            if isinstance(it, dict):
                it["source"] = c["path"]
                it["source_text"] = text
                out.append(it)
    return out, fails

synth_v2, fails_v2 = run_generation(
    QA_PROMPT_V2, final_docs, temperature=0.4
)
res_v2 = [validate_pair(s) for s in synth_v2]

p1 = sum(r["pass"] for r in results) / max(len(results), 1)
p2 = sum(r["pass"] for r in res_v2) / max(len(res_v2), 1)
print(f"\n기존 프롬프트 : 생성 {len(synth)}쌍, 통과율 {p1:.0%}")
print(f"개선 프롬프트 : 생성 {len(synth_v2)}쌍, 통과율 {p2:.0%}")

실패 유형 : {'표현복사': 2, '근거불일치': 3}

기존 프롬프트 : 생성 19쌍, 통과율 74%
개선 프롬프트 : 생성 27쌍, 통과율 70%


In [64]:
fail_types = Counter()
for r in results:
    for i in r["issues"]:
        fail_types[i.split(":")[0]] += 1
print("실패 유형 :", dict(fail_types))

QA_PROMPT_V2 = QA_PROMPT.replace(
    "6. JSON 배열로만 출력하세요. 설명을 붙이지 마세요.",
    """6. evidence는 본문에 존재하는 한 문장을 있는 그대로 복사하세요.
7. question은 일반 사용자의 질문 말투로 바꾸세요.
8. 설명, 인사말, 마크다운 코드블록 없이 오직 순수한 JSON 배열만 출력하세요.""",
)

def run_generation(prompt_tpl, docs, temperature=0.7):
    out, fails = [], []
    for c in docs:
        text = mask_pii(c["text"]) if c.get("policy") == "mask" else c["text"]
        if len(text) < MIN_GEN_CHARS:
            continue
        items, err = parse_json_array(
            gen(prompt_tpl.format(doc=fit_doc(text)), temperature=temperature))
        if items is None:
            fails.append((c["path"], err))
            continue
        for it in items:
            if isinstance(it, dict):
                it["source"] = c["path"]
                it["source_text"] = text
                out.append(it)
    return out, fails

synth_v2, fails_v2 = run_generation(
    QA_PROMPT_V2, final_docs)
res_v2 = [validate_pair(s) for s in synth_v2]

p1 = sum(r["pass"] for r in results) / max(len(results), 1)
p2 = sum(r["pass"] for r in res_v2) / max(len(res_v2), 1)
print(f"\n기존 프롬프트 : 생성 {len(synth)}쌍, 통과율 {p1:.0%}")
print(f"개선 프롬프트 : 생성 {len(synth_v2)}쌍, 통과율 {p2:.0%}")

실패 유형 : {'표현복사': 2, '근거불일치': 3}

기존 프롬프트 : 생성 19쌍, 통과율 74%
개선 프롬프트 : 생성 25쌍, 통과율 80%


<details class="lms-answer">
<summary>풀이 보기</summary>

**빈칸 정답 코드 (`### START CODE HERE ###` 영역)**

```python
QA_PROMPT_V2 = QA_PROMPT.replace(
    "6. JSON 배열로만 출력하세요. 설명을 붙이지 마세요.",
    ADDED_RULES
)
```

<details>
<summary>전체 완성 코드 보기 (복사·붙여넣기용)</summary>

```python
fail_types = Counter()
for r in results:
    for i in r["issues"]:
        fail_types[i.split(":")[0]] += 1
print("실패 유형 :", dict(fail_types))

QA_PROMPT_V2 = QA_PROMPT.replace(
    "6. JSON 배열로만 출력하세요. 설명을 붙이지 마세요.",
    """6. evidence 는 원문에서 문장을 글자 하나도 바꾸지 않고 복사해 넣으세요.
7. 답변에 숫자를 쓸 때는 원문에 있는 숫자만 쓰세요. 계산하거나 추정하지 마세요.
8. 질문에는 원문의 표현을 열 글자 이상 연속으로 쓰지 마세요. 뜻이 같은 다른 말로 바꾸세요.
9. JSON 배열로만 출력하세요. 설명이나 코드블록 표시를 붙이지 마세요."""
)

def run_generation(prompt_tpl, docs, temperature=0.7):
    out, fails = [], []
    for c in docs:
        text = mask_pii(c["text"]) if c.get("policy") == "mask" else c["text"]
        if len(text) < MIN_GEN_CHARS:
            continue
        items, err = parse_json_array(
            gen(prompt_tpl.format(doc=fit_doc(text)), temperature=temperature))
        if items is None:
            fails.append((c["path"], err))
            continue
        for it in items:
            if isinstance(it, dict):
                it["source"] = c["path"]
                it["source_text"] = text
                out.append(it)
    return out, fails

synth_v2, fails_v2 = run_generation(QA_PROMPT_V2, final_docs)
res_v2 = [validate_pair(s) for s in synth_v2]
p1 = sum(r["pass"] for r in results) / max(len(results), 1)
p2 = sum(r["pass"] for r in res_v2) / max(len(res_v2), 1)
print(f"\n기존 프롬프트 : 생성 {len(synth)}쌍, 통과율 {p1:.0%}")
print(f"개선 프롬프트 : 생성 {len(synth_v2)}쌍, 통과율 {p2:.0%}")
```

</details>

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

가장 효과가 큰 지시는 "**글자 하나도 바꾸지 말고 복사하라**"입니다. 모델은 인용하면서 자연스럽게 다듬는 버릇이 있어 근거 불일치가 생깁니다.

규칙 8(표현 복사 금지)과 규칙 6(원문 정확 인용)은 충돌하는 것처럼 보일 수 있습니다. 질문에는 다른 표현을, 근거에는 원문 그대로를 요구하므로 실제로는 충돌하지 않지만 모델이 헷갈릴 수 있습니다. **어느 항목에 적용되는 규칙인지를 문장 안에 넣어 두는 것**이 안전합니다.

통과율이 크게 오르지 않는다면 프롬프트가 아니라 **모델 크기가 한계**일 가능성을 봅니다. 3B 모델은 지시 항목이 아홉 개쯤 되면 뒤쪽 항목을 자주 흘립니다.

</details>

#### 트랙 4. 문서 버전 판별과 유효 문서 지정

Step 7에서 복무규정 2024년판과 2026년판이 유사도 0.7 기준에서 **근사 중복으로 잡혔습니다.** 이 둘을 중복이 아니라 **버전 관계**로 인식하게 만들어 Step 2의 카탈로그에 비워 두었던 다섯 필드를 채웁니다.

In [ ]:
# 실습 : 근사 중복 그룹에서 버전 관계를 판별하고 유효 문서를 결정하세요
#  - 연도 추출 함수(extract_year)와 출력 코드는 준비되어 있습니다.
#  - resolve_versions 안의 정렬 기준과 현행 여부(is_current), 대체 문서(superseded_by) 3줄만 채우세요.

def extract_year(c):
    """파일명과 본문 앞부분에서 연도를 추출합니다"""
    for src in (c["name"], c["text"][:300]):
        years = re.findall(r"(20\d{2})", src)
        if years:
            return max(int(y) for y in years)
    return None


def resolve_versions(group):
    """group : 같은 문서의 여러 버전. 정렬해 유효 기간을 부여합니다"""
    for c in group:
        c["doc_year"] = extract_year(c)

    ### START CODE HERE ### (버전 정렬과 현행 여부·대체 문서 지정 3곳을 채우세요)
    # 1) 연도(doc_year or 0), 수정시각(mtime), 본문길이(char_len) 순으로 오름차순 정렬합니다

    ordered = None  # TODO
    if ordered is None:
        return []

    for i, c in enumerate(ordered):
        nxt = ordered[i + 1] if i + 1 < len(ordered) else None
        c["doc_version"] = i + 1
        # 2) 다음 버전(nxt)이 없으면(None이면) 현행 문서(True)입니다
        c["is_current"] = None  # TODO: (nxt is None) 입력
        c["valid_from"] = f"{c['doc_year']}-01-01" if c["doc_year"] else None
        c["valid_until"] = (f"{nxt['doc_year']}-01-01"
                            if nxt and nxt["doc_year"] else None)
        # 3) 다음 버전(nxt)이 있으면 그 경로(nxt["path"])를 기록하고 없으면 None을 넣습니다
        c["superseded_by"] = None  # TODO
        ### END CODE HERE ###

        c["indexed"] = True          # 구버전도 인덱싱하되 필터로 구분합니다
        c.pop("near_duplicate_of", None)      # 중복이 아니라 버전이었습니다
    return ordered


# 완전 중복으로 걸러진 사본은 제외합니다. 같은 버전의 복사본이지 다른 버전이 아닙니다
policy_group = [c for c in catalog
                if "복무규정" in c["name"] and c["status"] == "parsed"]
for c in resolve_versions(policy_group):
    print(f"v{c['doc_version']} {c['name']:<24} 연도 {c['doc_year']} "
          f"현행 {str(c['is_current']):<5} 유효 {c['valid_from']} 에서 {c['valid_until']}")
    print(f"     대체 문서 : {c['superseded_by']}")

<details class="lms-answer">
<summary>풀이 보기</summary>

**빈칸 정답 코드 (`### START CODE HERE ###` 영역)**

```python
ordered = sorted(group, key=lambda c: (c["doc_year"] or 0, c["mtime"], c["char_len"]))
# ...
c["is_current"] = (nxt is None)
# ...
c["superseded_by"] = nxt["path"] if nxt else None
```

<details>
<summary>전체 완성 코드 보기 (복사·붙여넣기용)</summary>

```python
def extract_year(c):
    """파일명과 본문 앞부분에서 연도를 추출합니다"""
    for src in (c["name"], c["text"][:300]):
        years = re.findall(r"(20\d{2})", src)
        if years:
            return max(int(y) for y in years)
    return None

def resolve_versions(group):
    """group : 같은 문서의 여러 버전. 정렬해 유효 기간을 부여합니다"""
    for c in group:
        c["doc_year"] = extract_year(c)

    ordered = sorted(group, key=lambda c: (c["doc_year"] or 0, c["mtime"], c["char_len"]))

    for i, c in enumerate(ordered):
        nxt = ordered[i + 1] if i + 1 < len(ordered) else None
        c["doc_version"] = i + 1
        c["is_current"] = (nxt is None)
        c["valid_from"] = f"{c['doc_year']}-01-01" if c["doc_year"] else None
        c["valid_until"] = (f"{nxt['doc_year']}-01-01"
                            if nxt and nxt["doc_year"] else None)
        c["superseded_by"] = nxt["path"] if nxt else None
        c["indexed"] = True          # 구버전도 인덱싱하되 필터로 구분합니다
        c.pop("near_duplicate_of", None)      # 중복이 아니라 버전이었습니다
    return ordered

# 완전 중복으로 걸러진 사본은 제외합니다. 같은 버전의 복사본이지 다른 버전이 아닙니다
policy_group = [c for c in catalog
                if "복무규정" in c["name"] and c["status"] == "parsed"]
for c in resolve_versions(policy_group):
    print(f"v{c['doc_version']} {c['name']:<24} 연도 {c['doc_year']} "
          f"현행 {str(c['is_current']):<5} 유효 {c['valid_from']} 에서 {c['valid_until']}")
    print(f"     대체 문서 : {c['superseded_by']}")
```

</details>

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

먼저 **그룹을 고를 때 `status == "parsed"`만 썼습니다.** 파일명에 "복무규정"이 든 파일은 셋인데 그중 하나는 Step 7에서 완전 중복으로 걸러진 **같은 버전의 복사본**입니다. 복사본까지 넣으면 버전이 셋으로 세어져 유효 기간이 같은 버전 두 개가 생깁니다. 버전을 세기 전에 완전 중복을 먼저 지우는 순서가 여기서 쓸모 있습니다.

**구버전을 지우지 않고 유효 기간을 부여**합니다. `resolve_versions`가 `near_duplicate_of` 표시를 지운 것은 Step 7의 근사 중복 표시를 **버전 관계로 다시 해석한 것**입니다. 지우지 않고 표시만 해 두었기에 되돌릴 수 있었고 삭제했다면 되돌릴 방법이 없었습니다.

이렇게 하면 두 종류의 질문에 모두 답할 수 있습니다. "재택근무 며칠까지 되나요"에는 현행 버전으로 답하고 "재작년에는 어땠나요"에는 구버전으로 답합니다.

검색할 때는 기본 필터로 `is_current: true`를 걸고 사용자가 과거를 물을 때 필터를 해제합니다. 프라이빗 RAG 노트에서 배운 메타데이터 필터가 여기서 쓰입니다.

</details>

#### 트랙 5. 파인튜닝 데이터셋 변환
*(안내: 파인튜닝용 JSONL 변환과 데이터셋 카드 발행은 데이터셋 자동화 노트(D45) 본 실습 Step 7로 통합되어 진행합니다.)*

## Step 12. 실습 결과 점검과 정리

각자 수행한 두 트랙을 공유합니다.

**1. 선택한 트랙과 결과**

**2. 파이프라인 통계** : 원본 몇 건에서 시작해 몇 건이 어떤 사유로 빠지고 최종 몇 건이 남았는지

**3. 판단이 필요했던 지점** : 근사 중복 임계값, 개인정보 처리 방침, 버전 취급에서 어떤 판단을 내렸고 왜 그랬는지

**4. 합성 데이터 통과율과 개선 시도** : 생성 대비 통과 비율, 실패 유형, 프롬프트 개선 효과(트랙 3을 했다면)

처음으로 **지저분한 데이터**를 다뤘습니다. 지금까지는 늘 깔끔한 문서가 주어졌지만 실무에서는 이 정제 작업이 프로젝트 시간의 대부분을 차지합니다.

오늘 코드에서 반복해 나온 구조가 있습니다.

**저비용 방법으로 후보를 좁히고 정밀한 방법으로 검증합니다.** 해시로 완전 중복을 잡고 LSH로 근사 중복 후보를 좁히고 유사도를 다시 계산해 판정했습니다.

**삭제 대신 표시입니다.** 근사 중복도, 구버전도, 권한 문제 문서도, 줄 단위 정제 규칙에 걸린 줄도 지우지 않고 표시만 했습니다. 그래서 심화 트랙 3에서 "중복인 줄 알았는데 버전이었다"를 되돌릴 수 있었습니다.

**실패는 예외가 아니라 정상 경로입니다.** 인코딩 추측이 틀려도 파서가 실패해도 모델이 JSON을 깨뜨려도 파이프라인은 멈추지 않았고 사유를 남겼습니다.

파이프라인 노트에서는 오늘 다룬 단계들을 포함한 **열 단계 파이프라인 전체**를 하나의 실행 가능한 시스템으로 조립합니다. 오늘 구현한 전처리 로직이 그 파이프라인의 각 단계가 됩니다.

## 마무리 과제. 사내 문서 저장소 전처리 파이프라인 구축

여러분이 준비한 문서 모음으로 수집과 정제 파이프라인을 구축합니다.

**1단계. 저장소 준비**

1. 문서 15건 이상을 모읍니다. 형식이 최소 3종 이상 섞여 있어야 합니다.  
2. 의도적으로 다음을 포함시킵니다. 완전 중복 1건 이상, 근사 중복 1쌍 이상, 빈 문서 1건, 민감 정보 포함 문서 1건.

**2단계. 파이프라인 구현**

3. 카탈로그를 작성합니다. 메타데이터 항목 6개 이상. **버전과 유효 여부, 라이선스와 이용 조건(교재 4.4.2)을 담을 필드를 반드시 포함**합니다.  
4. 개인정보 탐지와 처리 방침을 구현합니다. 위험도에 따라 제외와 마스킹을 구분해야 합니다.  
5. 형식별 파서를 붙이고 이론 6.1절의 지표를 문서마다 측정합니다.  
6. 필터링과 중복 제거를 구현합니다. **각 규칙이 걸러낸 건수와 사유를 기록**해야 합니다.

**3단계. 합성 데이터 생성**

7. 정제된 문서로 QA 쌍을 생성합니다. 목표 30쌍 이상.  
8. 검증 함수를 구현하고 통과율을 측정합니다.  
9. (선택) 프롬프트를 한 번 이상 개선해 통과율 변화를 기록합니다. 심화 트랙 2를 수행했다면 그 결과를 활용합니다.

**4단계. 리포트**

- **파이프라인 통계표**

<div style="font-size: 18px; line-height: 1.6;">

| 단계 | 입력 | 출력 | 제외 | 주요 사유 |
|---|---|---|---|---|
| 파싱 | | | | |
| 필터링 | | | | |
| 완전 중복 | | | | |
| 근사 중복 | | | | |

</div>

- **임계값 결정 근거** : 근사 중복 임계값을 얼마로 정했고 왜 그랬는지. 값을 바꿔 봤을 때 무엇이 달라졌는지  
- **잘못 걸러진 문서** : 필터가 제외했지만 실제로는 쓸 만했던 문서가 있는지. 있다면 기준을 어떻게 고칠지  
- **정규식 오탐 확인** : 패턴마다 몇 건이 걸렸는지와 걸린 것의 표본. 엉뚱한 것을 잡고 있는 패턴이 있었는지  
- **합성 데이터 실패 분석** : 실패 유형별 건수와 원인, 9번을 했다면 개선 시도와 결과  
- **권한 문제 문서 처리 방침** : 기밀 문서를 어떻게 다룰지에 대한 설계

**평가 기준**

파이프라인이 동작하는지가 아니라 **판단과 기록**으로 평가합니다. 임계값을 왜 그렇게 정했는지, 걸러낸 것을 어떻게 추적할 수 있는지, 잘못 걸러진 것을 발견했는지를 봅니다. 통계표에 사유가 비어 있으면 감사 불가능한 파이프라인입니다.